# Thesis figures

This notebook draws every figure of the thesis, in thesis order, from the stored outputs of the pipeline stages (JSON, NPZ and marker files).
It fits nothing and changes no number. Use "Restart and run all".

- Outputs go to `parquet/model/figures/notebook/` in the data folder (`THESIS_DATA_ROOT`, see `pipeline/config.py`): for each figure a PNG at 300 dpi, a PDF, an SVG and a CSV file with the plotted values, and `captions.md` with the figure numbers and captions.
- Where a thesis table holds a plotted number, the notebook computes the number again from the stored arrays and compares it with the JSON file that the table quotes. A difference stops the notebook with a message.
- When the evidence-removal stage has not run, its figure is skipped with a message, and the other figures are drawn.
- The thesis document links the PNG files.

In [ ]:
# Setup: paths, the model package, the figure style and the arm colours.
import csv
import functools
import io
import json
import math
import os
import struct
import sys
import textwrap
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib as mpl
import numpy as np

T0 = time.time()
NOTEBOOKS = Path.cwd()  # the repository root, which holds this notebook and the pipeline package
if str(NOTEBOOKS) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS))
from pipeline import gam  # the model's tables and their centring
from pipeline.config import FROZEN  # the frozen counts of the data
from pipeline.config import MODEL_DIR, ROOT, SCRATCH_DIR  # the data folder (THESIS_DATA_ROOT)

OUT = MODEL_DIR / "figures" / "notebook"
OUT.mkdir(parents=True, exist_ok=True)
DEMO_USER = os.environ.get("DEMO_USER")  # a folder under parquet/model/demo; None takes the only one there

try:
    get_ipython  # defined inside Jupyter
    IN_NOTEBOOK = True
except NameError:
    IN_NOTEBOOK = False
    mpl.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager, patheffects, ticker
from matplotlib import scale as mscale
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Patch

MM = 1 / 25.4
WIDTH = 150 * MM  # A4 minus 30 mm margins (Capstone Project Module Handbook, 4.3)
DPI = 300
FORMATS = ("png", "pdf", "svg")
META = {"png": {"Software": None},  # no tool names or times in the files, so a rerun gives the same bytes
        "pdf": {"Creator": None, "Producer": None, "CreationDate": None},
        "svg": {"Creator": None, "Date": None}}

INK = "#222222"
INK_2 = "#555555"
MUTED = "#8c8c8c"
GRID = "#e4e4e4"
RULE = "#c4c4c4"
FILL = "#f4f4f4"
SURFACE = "#ffffff"

# One fixed colour per arm, in the same order in every figure.
ARMS = {
    "gam": ("GAM", "#2a78d6"),
    "knn_same": ("ItemKNN, same-artist list", "#eb6834"),
    "als_d": ("ALS, 256 factors", "#1baf7a"),
    "artist_only": ("Artist terms only", "#eda100"),
    "cold_gam": ("Cold-track GAM", "#e87ba4"),
    "knn_both": ("ItemKNN, both lists", "#008300"),
    "als_p6": ("ALS, 64 factors", "#6250d6"),
    "popularity": ("Popularity", "#e34948"),
}
RAMP = ["#86b6ef", "#2a78d6", "#104281"]  # one hue, light to dark, for ordered groups
CONCEPT_LABEL = {
    "colisten_same_artist": "Same-artist co-listening",
    "artist_affinity": "Artist affinity",
    "artist_tag_affinity": "Artist-tag affinity",
    "tag_affinity": "Tag affinity",
    "genre_affinity": "Genre affinity",
    "audio_fit": "Audio fit",
}
REASON_LABEL = {"artist": "Artist", "artist_tag_affinity": "Artist tags", "tag_affinity": "Track tags",
                "genre_affinity": "Genres", "audio_fit": "Audio fit"}
LETTERS = "abcdefghijklmnop"


def label(arm):
    return ARMS[arm][0]


def color(arm):
    return ARMS[arm][1]


for path in font_manager.findSystemFonts():
    if "LiberationSans" in path:
        font_manager.fontManager.addfont(path)
FONT = ("Liberation Sans" if any(f.name == "Liberation Sans" for f in font_manager.fontManager.ttflist)
        else "DejaVu Sans")

mpl.rcParams.update({
    "font.family": FONT,
    "font.size": 8,
    "axes.titlesize": 8.5,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.labelsize": 8,
    "axes.edgecolor": INK_2,
    "axes.labelcolor": INK,
    "axes.linewidth": 0.6,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": GRID,
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "xtick.color": INK_2,
    "ytick.color": INK_2,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "legend.frameon": False,
    "text.color": INK,
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "figure.dpi": 100,
    "axes.unicode_minus": False,
    "svg.fonttype": "none",
    "svg.hashsalt": "thesis-figures",
    "pdf.fonttype": 42,
})
print("Data folder:", ROOT)
print("Outputs:", OUT)
print(f"Font: {FONT}, matplotlib {mpl.__version__}, numpy {np.__version__}")

## Helpers

Checks, file access, the statistics of the stages (the same formulas, so a recomputed number can equal its stored value), saving and progress.

In [ ]:
# Helpers used by every figure cell.
class FigureCheckError(Exception):
    """A source is missing, or a plotted number differs from its source."""


def check(ok, message):
    if not ok:
        raise FigureCheckError(message)


def load_json(name, folder=MODEL_DIR):
    path = Path(folder) / name
    check(path.is_file(), f"{path} is missing")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def load_npz(name, keys, folder=MODEL_DIR):
    """The named arrays of an NPZ file, and only those."""
    path = Path(folder) / name
    check(path.is_file(), f"{path} is missing")
    with np.load(path) as z:
        missing = [k for k in keys if k not in z.files]
        check(not missing, f"{path.name} holds no array {missing}")
        return {k: np.asarray(z[k]) for k in keys}


def num(x):
    return math.nan if x is None else float(x)


def gap_of(a, b):
    """The largest absolute difference of two arrays. NaN equals only NaN."""
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    check(a.shape == b.shape, f"arrays of shapes {a.shape} and {b.shape} cannot be compared")
    if a.size == 0:
        return 0.0
    na, nb = np.isnan(a), np.isnan(b)
    d = np.where(na | nb, 0.0, np.abs(np.where(na, 0.0, a) - np.where(nb, 0.0, b)))
    return float(np.inf) if np.any(na ^ nb) else float(d.max())


class Checks:
    """The checks of one figure: each compares numbers computed here with a source file and stops on a difference."""

    def __init__(self):
        self.n, self.worst, self.files = 0, 0.0, []

    def _note(self, source, n):
        self.n += n
        if source not in self.files:
            self.files.append(source)

    def close(self, have, want, what, source, tol=1e-12):
        g = gap_of(have, want)
        check(g <= tol, f"{what}: the value computed here differs from {source} by {g:.2e} (tolerance {tol:.0e})")
        self.worst = max(self.worst, g)
        self._note(source, int(np.size(have)))

    def same(self, have, want, what, source):
        check(have == want, f"{what}: {have!r} computed here, {want!r} in {source}")
        self._note(source, 1)

    def summary(self):
        if not self.n:
            return "No table quotes these values."
        return f"Checks: {self.n:,} values equal {', '.join(self.files)} (largest gap {self.worst:.1e})."


Z95 = 1.959963984540054  # the normal quantile of every stage
QUANTILES = (5, 25, 50, 75, 95)


def pick_metrics(ranks):
    """Per-pick metrics from exact ranks, as P14 computes them."""
    r = np.asarray(ranks, dtype=np.float64)
    gain = 1.0 / np.log2(r + 1.0)
    return {"ndcg_at_100": np.where(r <= 100, gain, 0.0),
            "ndcg_at_10": np.where(r <= 10, gain, 0.0),
            "recall_at_10": (r <= 10).astype(np.float64),
            "recall_at_100": (r <= 100).astype(np.float64),
            "mrr": 1.0 / r}


def listener_metrics(ranks):
    return {k: v.mean(axis=1) for k, v in pick_metrics(ranks).items()}


def interval(x):
    """The mean over listeners with a 95 percent normal interval."""
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    m = float(x.mean())
    se = float(x.std(ddof=1) / math.sqrt(n)) if n > 1 else math.nan
    return {"mean": m, "lo": m - Z95 * se, "hi": m + Z95 * se}


def paired(a, b):
    """The mean of a - b over listeners with its 95 percent interval."""
    return interval(np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64))


def cluster_mean(x, mask):
    """The mean of per-pick values over the picks in mask, with the interval for listener clusters."""
    x = np.asarray(x, dtype=np.float64)
    mask = np.asarray(mask, dtype=bool)
    n_u = mask.sum(axis=1).astype(np.float64)
    tot = np.where(mask, x, 0.0).sum(axis=1)
    n = float(n_u.sum())
    g = int(np.count_nonzero(n_u))
    if n == 0:
        return {"mean": math.nan, "lo": math.nan, "hi": math.nan, "picks": 0}
    m = float(tot.sum() / n)
    se = math.nan
    if g > 1:
        r = (tot - m * n_u)[n_u > 0]
        se = math.sqrt(g / (g - 1) * float(np.sum(r * r))) / n
    return {"mean": m, "lo": m - Z95 * se, "hi": m + Z95 * se, "picks": int(n)}


def rank_quantiles(r):
    """The 5th to 95th percentiles of ranks, each a rank that occurs (inverted empirical distribution)."""
    r = np.asarray(r, dtype=np.int64)
    if not r.size:
        return [None] * len(QUANTILES)
    return [int(v) for v in np.percentile(r, QUANTILES, method="inverted_cdf")]


def triple(d):
    """(mean, lo, hi) of a stored interval."""
    return [num(d["mean"]), num(d["lo"]), num(d["hi"])]


def words(n):
    """A count in words up to ten, as the thesis writes them."""
    names = "zero one two three four five six seven eight nine ten".split()
    return names[n] if 0 <= n <= 10 else f"{n:,}"


def panel(ax, k, name=None):
    """The panel letter, with a short name when panels repeat one chart."""
    ax.set_title(f"({LETTERS[k]})" + (f" {name}" if name else ""), loc="left", fontweight="bold", fontsize=8)


def label_bars(ax, bars, fmt, pad=2, size=6.5):
    for bar in bars:
        w = bar.get_width()
        ax.annotate(fmt(w), (w, bar.get_y() + bar.get_height() / 2), xytext=(pad, 0), textcoords="offset points",
                    va="center", ha="left", fontsize=size, color=INK)


def rank_ticks(ax, lo=1, hi=1e7, every=1):
    """Log axis of ranks with ticks 1, 10, 100, 1k and so on (every k-th power of ten)."""
    ax.set_xscale("log")
    ticks = [10.0 ** k for k in range(int(math.floor(math.log10(lo))), int(math.ceil(math.log10(hi))) + 1, every)]
    names = {1: "1", 10: "10", 100: "100", 1e3: "1k", 1e4: "10k", 1e5: "100k", 1e6: "1M", 1e7: "10M", 1e8: "100M"}
    ax.xaxis.set_major_locator(ticker.FixedLocator(ticks))
    ax.xaxis.set_major_formatter(ticker.FixedFormatter([names.get(t, f"{t:g}") for t in ticks]))
    ax.xaxis.set_minor_locator(ticker.NullLocator())


def csv_text(header, rows):
    buf = io.StringIO()
    w = csv.writer(buf, lineterminator="\n")
    w.writerow(header)
    for r in rows:
        w.writerow(["" if x is None else (repr(float(x)) if isinstance(x, (float, np.floating)) else str(x))
                    for x in r])
    return buf.getvalue()


def png_size(path):
    with open(path, "rb") as f:
        head = f.read(24)
    check(head[:8] == b"\x89PNG\r\n\x1a\n", f"{path.name} is no PNG file")
    return struct.unpack(">II", head[16:24])


def save(fig, name, header, rows):
    """The figure as PNG, PDF and SVG, and the plotted values as CSV. Each file is read back."""
    for ext in FORMATS:
        fig.savefig(OUT / f"{name}.{ext}", format=ext, dpi=DPI, metadata=META[ext])
    text = csv_text(header, rows)
    with open(OUT / f"{name}.csv", "w", newline="", encoding="utf-8") as f:
        f.write(text)
    with open(OUT / f"{name}.csv", newline="", encoding="utf-8") as f:
        check(f.read() == text, f"{name}.csv does not read back")
    want = round(WIDTH * DPI)
    w, _ = png_size(OUT / f"{name}.png")
    check(abs(w - want) <= 1, f"{name}.png is {w} px wide, not {want} (150 mm at {DPI} dpi)")
    check((OUT / f"{name}.pdf").read_bytes()[:4] == b"%PDF", f"{name}.pdf is no PDF file")
    ET.parse(OUT / f"{name}.svg")
    if IN_NOTEBOOK:
        plt.show()
    plt.close(fig)


FIGURES = (("3.1", "fig_pipeline"), ("3.2", "fig_bridge_tiers"), ("3.3", "fig_coverage"),
           ("4.1", "fig_loss_forms"), ("4.2", "fig_shapes"), ("4.3", "fig_artist_suppression"),
           ("6.1", "fig_accuracy"), ("6.2", "fig_slices"), ("6.3", "fig_beyond"), ("6.4", "fig_cold_start"),
           ("6.5", "fig_reasons"), ("6.6", "fig_evidence_removal"), ("6.7", "fig_tree_audit"),
           ("6.8", "fig_demo_match"), ("6.9", "fig_explanations"))
NUMBER = {name: n for n, name in FIGURES}
RESULTS = {}


def _line(name, text):
    k = [n for _, n in FIGURES].index(name) + 1
    print(f"[{k:2d}/{len(FIGURES)}] Figure {NUMBER[name]} {name}: {text}")


def finish(name, t_start, checks, caption, sources, values=""):
    """Record the caption and print the figure's progress line."""
    RESULTS[name] = {"status": "drawn", "caption": caption, "sources": sources, "values": values}
    _line(name, f"4 files in {time.time() - t_start:.1f} s (total {time.time() - T0:.1f} s). {checks.summary()}")


def skip(name, reason):
    RESULTS[name] = {"status": "skipped", "reason": reason}
    _line(name, f"skipped: {reason}")

## Shared sources

The final model and P14's results, read once and used by several figures.

- `p13c_final.json` names the final model's files: `p13c_<tag>_model.json` (terms, parameters, stored tables, the C1 bands and the importance) and `p13c_<tag>_arrays.npz` (the 100 listener and 50 two-way bootstrap replicates).
- `p13a_model.json` holds the layout constants that turn parameters into tables, and the training positives' shape quantiles.
- `scratch/p13c_<tag>/fits.json`, the saved work of the P13c run, is the only file that holds the background means that centre the replicates.
- `p13d_model_set.npz` holds the tables of the free fit beside the bounded one.

In [ ]:
# Shared sources: the final model (p13c_final.json and the files it names) and P14 (p14.json, p14.npz).
P14_ARMS = ("gam", "artist_only", "cold_gam", "knn_same", "knn_both", "als_d", "als_p6", "popularity")
PICK_TYPES = ("test_last", "test_first")
METRICS = ("ndcg_at_100", "ndcg_at_10", "recall_at_10", "recall_at_100", "mrr")
SCHEMES = ("listener", "twoway")
BAND = (0.025, 0.975)
POINTS = ("p5", "p25", "p50", "p75", "p95", "zero", "missing")
CENTER_TOL = 1e-12


@functools.lru_cache(maxsize=None)
def p14_doc():
    doc = load_json("p14.json")
    check(sorted(doc["arms"]) == sorted(P14_ARMS), f"p14.json holds the arms {sorted(doc['arms'])}")
    return doc


@functools.lru_cache(maxsize=None)
def p14_arrays():
    return load_npz("p14.npz", [f"rank_{a}" for a in P14_ARMS] + ["pop_centile", "neighbour", "candidate",
                                                                    "shown_d1", "shown", "reason_value"])


@functools.lru_cache(maxsize=None)
def final_model():
    """The final model, its centred bootstrap replicates and the free fit, checked against the stored tables."""
    ptr = load_json("p13c_final.json")
    rec, files = ptr["model"], ptr["files"]
    doc = load_json(files["results"])
    m = doc["model"]
    a_doc = load_json("p13a_model.json")
    terms = [gam.Term.from_json(t) for t in m["terms"]]
    names = [t.name for t in terms]
    meta, refm = a_doc["layout"], a_doc["reference_means"]
    lay = gam.Layout(terms, {n: int(meta["n_zero"][n]) for n in names}, {n: int(meta["n_miss"][n]) for n in names},
                     [n for n in meta["neutral"] if n in names],
                     {n: np.asarray(v) for n, v in refm["data"].items() if n in names},
                     {n: np.asarray(v) for n, v in refm["dev"].items() if n in names})
    theta = np.asarray(m["theta"], dtype=np.float64)
    raw_gap = gap_of(lay.tables(theta), m["tables_raw"])
    check(raw_gap <= 1e-12, f"the final model's layout, rebuilt from p13a_model.json, gives raw tables that differ "
                            f"from {files['results']} by {raw_gap:.1e}")
    tag = rec["tag"] if rec["changed"] else None
    work = SCRATCH_DIR / (f"p13c_{tag}" if tag else "p13c")
    saved = (load_json("fits.json", work).get("means") or {}).get(f"wide:{m['level']}")
    check(saved is not None, f"{work / 'fits.json'} holds no background means of level {m['level']}. The P13c run "
                             f"saves them there, and the bands of the shapes need them")
    means = {n: np.asarray(saved[n], dtype=np.float64) for n in names}
    stored = np.asarray(m["tables"], dtype=np.float64)
    center_gap = gap_of(gam.center(lay.tables(theta), lay, means)[0], stored)
    check(center_gap <= CENTER_TOL, f"the final model centred with the saved background means differs from its "
                                    f"stored tables by {center_gap:.1e}")
    z = load_npz(files["arrays"], [f"theta_boot_{s}" for s in SCHEMES])
    boot = {s: np.array([gam.center(lay.tables(th), lay, means)[0] for th in z[f"theta_boot_{s}"]])
            for s in SCHEMES}
    zs = load_npz("p13d_model_set.npz", ["names", "tables", "members"])
    fits = [str(x) for x in zs["names"].tolist()]
    check(rec["fit_name"] in fits and np.array_equal(zs["tables"][fits.index(rec["fit_name"])], stored),
          "p13d_model_set.npz holds other tables for the final fit than the final model's")
    family, _, rest = rec["fit_name"].partition("|")
    free = None
    if family == "bounded":
        twin = f"free|{rest}"
        check(twin in fits, f"p13d_model_set.npz holds no fit {twin}")
        free = {"name": twin, "tables": np.asarray(zs["tables"][fits.index(twin)], dtype=np.float64)}
    return {"rec": rec, "files": files, "doc": doc, "lay": lay, "names": names, "stored": stored, "boot": boot,
            "free": free, "bounded": list(rec.get("bounded") or []), "shape_q": a_doc["shape_quantiles"],
            "audit": a_doc["audit"], "members": [str(x) for x in zs["members"].tolist()],
            "raw_gap": raw_gap, "center_gap": center_gap}


def c1_row(term, point):
    """The stored C1 row of a term at a point (value and bands), or None."""
    rows = final_model()["doc"]["bootstrap"]["bands"].get(term) or []
    return next((r for r in rows if r["point"] == point), None)

## Figure 3.1: the pipeline

The stages of the runner in five phases, between a box for the data and a box for the plain scripts and this notebook.
A stage is grey when no stage of the evaluation phase depends on it, directly or through other stages.
Source: the stage markers in `markers/` (each holds the stage name and the markers of its inputs). The phases and the short labels are a layout choice of this figure.

In [ ]:
# Figure 3.1: the stages and their upstream stages, from the markers in markers/.
PHASES = (
    ("Preparation", ("p1_audit", "p2_split", "p25_train", "p3_sampling")),
    ("Features and baselines", ("p4_features", "p5_eval_baseline", "p6_als", "p7_graph", "p8_fusion_features",
                                "p9_cooccurrence", "p9b_colisten", "p10_profiles", "als_phase_d",
                                "itemknn_select")),
    ("Training pairs and model fits", ("p11_fusion_pairs", "p12_pair_features", "p13a_fit", "p13b_arms",
                                       "p11b_more_pairs", "p12b_more_features", "p13b2_loss", "gam_knn_slices",
                                       "gam_refit")),
    ("Stability and the frozen model", ("p13c_stability", "p13d_bounds", "p13c_final")),
    ("Evaluation", ("p14_cold_gam", "p14_eval", "cold_start_curve", "beyond_accuracy", "evidence_removal")),
)
STAGE_TEXT = {
    "p1_audit": ("P1", "corpus audit"), "p2_split": ("P2", "train, validation, test split"),
    "p25_train": ("P2.5", "training table, statistics"), "p3_sampling": ("P3", "sampling tables"),
    "p4_features": ("P4", "item feature matrices"), "p5_eval_baseline": ("P5", "evaluation harness, popularity"),
    "p6_als": ("P6", "ALS baseline"), "p7_graph": ("P7", "item-item relational graph"),
    "p8_fusion_features": ("P8", "extra fusion features"), "p9_cooccurrence": ("P9", "item co-occurrence"),
    "p9b_colisten": ("P9b", "exact co-listening neighbours"), "p10_profiles": ("P10", "user profile vectors"),
    "als_phase_d": ("ALS-D", "ALS phase D tuning"), "itemknn_select": ("KNN", "ItemKNN baseline"),
    "p11_fusion_pairs": ("P11", "training pairs"), "p12_pair_features": ("P12", "concept features of the pairs"),
    "p13a_fit": ("P13a", "audit and base GAM fit"), "p13b_arms": ("P13b", "experiment arms"),
    "p11b_more_pairs": ("P11b", "more negatives"), "p12b_more_features": ("P12b", "their concept features"),
    "p13b2_loss": ("P13b2", "loss form, negatives"), "gam_knn_slices": ("Diag", "GAM against ItemKNN slices"),
    "gam_refit": ("Refit", "GAM refit, 2 × 2 design"), "p13c_stability": ("P13c", "bootstrap stability"),
    "p13d_bounds": ("P13d", "bounded content shapes"), "p13c_final": ("P13c", "stability of the final model"),
    "p14_cold_gam": ("P14", "cold-track GAM"), "p14_eval": ("P14", "evaluation on the eval listeners"),
    "cold_start_curve": ("Curve", "cold-start curve"), "beyond_accuracy": ("Beyond", "novelty, coverage, diversity"),
    "evidence_removal": ("Removal", "evidence removal of the reasons"),
}
DATA_BOX = ("Data and ETL scripts (outside the runner)",
            "MLHD+ listening histories, the MusicBrainz database and the Spotify catalogue, converted to Parquet "
            "and bridged into one item space.")
SCRIPTS_TITLE = "Plain scripts and the figures notebook, on the stored outputs"
SCRIPTS = (("lb_demo.py", "pipeline/lb_demo.py", "ListenBrainz demo"),
           ("lb_outside.py", "pipeline/lb_outside.py", "tracks outside MLHD"),
           ("tree_audit.py", "pipeline/tree_audit.py", "tree-GAM audit"),
           ("figures.py", "pipeline/figures.py", "thesis tables"),
           ("notebook", "Thesis Figures.ipynb", "thesis figures"))


def pipeline_data():
    up = {}
    for p in sorted((MODEL_DIR / "markers").glob("*.json")):
        mk = json.loads(p.read_text(encoding="utf-8"))
        up[mk.get("stage") or p.stem] = sorted(k.split(":", 1)[1] for k in (mk.get("inputs") or {})
                                               if k.startswith("marker:"))
    names = list(up)
    check(names, f"{MODEL_DIR / 'markers'} holds no stage marker")
    placed = [n for _, members in PHASES for n in members]
    check(sorted(placed) == sorted(names) and len(set(placed)) == len(placed),
          f"PHASES does not hold every stage marker exactly once: missing {sorted(set(names) - set(placed))}, "
          f"without a marker {sorted(set(placed) - set(names))}")
    phase = {n: k for k, (_, members) in enumerate(PHASES) for n in members}
    for n, ups in up.items():
        unknown = [u for u in ups if u not in phase]
        check(not unknown, f"the marker of {n} names unknown stages {unknown}")
        late = [u for u in ups if phase[u] > phase[n]]
        check(not late, f"{n} depends on {late}, which sit in a later phase")
    used = set(PHASES[-1][1])
    stack = list(used)
    while stack:
        for u in up[stack.pop()]:
            if u not in used:
                used.add(u)
                stack.append(u)
    missing = [rel for _, rel, _ in SCRIPTS if not (NOTEBOOKS / rel).is_file()]
    check(not missing, f"the scripts {missing} are not in {NOTEBOOKS}")
    boxes = [{"title": DATA_BOX[0], "text": DATA_BOX[1], "items": []}]
    for title, members in PHASES:
        boxes.append({"title": title, "text": None,
                      "items": [(n,) + STAGE_TEXT[n] + (n in used,) for n in members]})
    boxes.append({"title": SCRIPTS_TITLE, "text": None, "items": [(f, f, txt, True) for f, _, txt in SCRIPTS]})
    unused = [n for _, members in PHASES for n in members if n not in used]
    return {"boxes": boxes, "upstream": up, "unused": unused, "n_stages": len(names)}


def draw_pipeline(P):
    W = 150.0
    cols, row_h, head, pad, gap = 3, 4.1, 6.0, 2.2, 5.0
    heights = []
    for b in P["boxes"]:
        lines = textwrap.wrap(b["text"], 112) if b["text"] else []
        heights.append(head + pad + row_h * (math.ceil(len(b["items"]) / cols) + len(lines)))
    H = sum(heights) + gap * (len(heights) - 1) + 9.0
    fig = plt.figure(figsize=(W * MM, H * MM))
    ax = fig.add_axes((0.0, 0.0, 1.0, 1.0))
    ax.set_xlim(0.0, W)
    ax.set_ylim(H, 0.0)
    ax.axis("off")
    y = 1.0
    colw = (W - 8.0) / cols
    for i, (b, h) in enumerate(zip(P["boxes"], heights)):
        runner_box = 0 < i < len(P["boxes"]) - 1
        ax.add_patch(FancyBboxPatch((1.0, y), W - 2.0, h, boxstyle="round,pad=0,rounding_size=1.6",
                                    fc=FILL if runner_box else SURFACE, ec=RULE, lw=0.7))
        ax.text(3.5, y + 3.6, b["title"], fontsize=8.0, fontweight="bold", va="center", ha="left", color=INK)
        yy = y + head + 1.4
        for line in (textwrap.wrap(b["text"], 112) if b["text"] else []):
            ax.text(3.5, yy, line, fontsize=6.8, va="center", ha="left", color=INK_2)
            yy += row_h
        for k, (name, sid, desc, used) in enumerate(b["items"]):
            r, c = divmod(k, cols)
            x0, y0 = 3.5 + c * colw, yy + r * row_h
            ink = INK if used else MUTED
            ax.text(x0, y0, sid, fontsize=6.6, va="center", ha="left", color=ink, fontweight="bold")
            ax.text(x0 + (17.0 if name == sid else 11.5), y0, desc, fontsize=6.6, va="center", ha="left",
                    color=INK_2 if used else MUTED)
        y += h
        if i < len(P["boxes"]) - 1:
            ax.add_patch(FancyArrowPatch((W / 2.0, y + 0.5), (W / 2.0, y + gap - 0.5), arrowstyle="-|>",
                                         mutation_scale=7.0, color=INK_2, lw=0.8))
            y += gap
    ax.text(1.5, H - 3.5, "Grey: no evaluation stage depends on these stages (not used by the final model). "
            "Arrows: the order of the phases.", fontsize=6.3, va="center", ha="left", color=MUTED)
    return fig


t = time.time()
P = pipeline_data()
rows = [[b["title"], name, sid, desc, "yes" if used else "no", " ".join(P["upstream"].get(name, []))]
        for b in P["boxes"] for name, sid, desc, used in b["items"]]
save(draw_pipeline(P), "fig_pipeline", ["box", "name", "id", "label", "used_by_final_model", "upstream"], rows)
unused_ids = ", ".join(STAGE_TEXT[n][0] for n in P["unused"]) or "none"
finish("fig_pipeline", t, Checks(),
       f"The pipeline from the raw sources to explained recommendations. Each box is a phase, and each row in it "
       f"a stage of the runner, which reads only the outputs of earlier stages. Grey stages ({unused_ids}) are not "
       f"used by the final model. The last box holds the plain scripts and the notebook that draws the figures "
       f"from the stored outputs.",
       f"the markers of the {P['n_stages']} runner stages (markers/)")

## Figure 3.2: the Spotify bridge

Share of items and share of listens per tier of the bridge. No stage output holds these counts, so they are constants from Table B.1 of the thesis.
The cell checks them against the item count of `id_spaces.json` and the frozen counts of `pipeline/config.py`.

In [ ]:
# Figure 3.2: the bridge tiers, from Table B.1 of the thesis (the lines on item_spotify_map,
# item_spotify_fuzzy and the per-tier listen decomposition), checked against id_spaces.json.
TIERS = (  # (tier, items, share of all listens in percent)
    ("ISRC and link agree", 99_559, 26.02),
    ("ISRC", 910_895, 41.66),
    ("Spotify link", 9_788, 0.97),
    ("Name and duration", 2_393_709, 20.05),
    ("Name only (display)", 637_398, 5.82),
    ("Unmatched", 2_586_033, 5.48),
)

t = time.time()
chk = Checks()
n_items = int(load_json("id_spaces.json")["items"]["distinct"])
chk.same(sum(x[1] for x in TIERS), n_items, "items over all tiers", "id_spaces.json")
chk.same(sum(x[1] for x in TIERS), FROZEN["items"], "items over all tiers", "pipeline/config.py")
chk.same(sum(x[1] for x in TIERS[:3]), FROZEN["map_items"], "items of the identifier tiers", "pipeline/config.py")
chk.same(TIERS[3][1], FROZEN["fuzzy_name_duration"], "items of the name and duration tier", "pipeline/config.py")
chk.same(sum(x[1] for x in TIERS[:5]), FROZEN["bridge_items"], "items of the matched tiers", "pipeline/config.py")
check(abs(sum(x[2] for x in TIERS) - 100.0) < 0.005, "the listen shares of the tiers do not sum to 100 percent")

items = [100 * x[1] / n_items for x in TIERS]
listens = [x[2] for x in TIERS]
y = np.arange(len(TIERS))[::-1]
h = 0.38
fig, ax = plt.subplots(figsize=(WIDTH, 62 * MM), layout="constrained")
b1 = ax.barh(y + h / 2, items, height=h, color=color("knn_same"), edgecolor=SURFACE, linewidth=0.6,
             label="Share of items")
b2 = ax.barh(y - h / 2, listens, height=h, color=color("gam"), edgecolor=SURFACE, linewidth=0.6,
             label="Share of listens")
label_bars(ax, b1, lambda v: f"{v:.1f}%")
label_bars(ax, b2, lambda v: f"{v:.1f}%")
ax.set_yticks(y, [x[0] for x in TIERS])
ax.set_xlabel("Percent")
ax.set_xlim(0, 10 * math.ceil(max(items + listens) * 1.18 / 10))
ax.grid(axis="y", visible=False)
fig.legend(loc="outside lower center", ncol=2)
save(fig, "fig_bridge_tiers", ["tier", "items", "share_items_pct", "share_listens_pct"],
     [[x[0], x[1], i, s] for x, i, s in zip(TIERS, items, listens)])
id_items = sum(items[:3])
id_listens = sum(listens[:3])
finish("fig_bridge_tiers", t, chk,
       f"Tiers of the Spotify bridge, from the most certain rule (top) to the least certain, with each tier's share "
       f"of the {n_items:,} items and of all listens. The identifier tiers carry {id_listens:.1f}% of the listens but "
       f"only {id_items:.1f}% of the items, because popular tracks are the ones that carry identifiers.",
       "Table B.1 of the thesis, checked against id_spaces.json and pipeline/config.py",
       f"identifier tiers {id_items:.2f}% of items and {id_listens:.2f}% of listens; matched tiers "
       f"{sum(items[:5]):.2f}% of items and {sum(listens[:5]):.2f}% of listens")

## Figure 3.3: evidence coverage follows popularity

Share of rows with data for each concept of the final model: random catalogue tracks (uniform negatives), popularity-weighted tracks (p_pop negatives) and played tracks (training positives).
Source: the audit before the first fit in `p13a_model.json` (`audit.missing.concepts`).

In [ ]:
# Figure 3.3: rows with data per concept, from p13a_model.json.
COVERAGE_ORDER = ("colisten_same_artist", "artist_affinity", "artist_tag_affinity", "tag_affinity",
                  "genre_affinity", "audio_fit")

t = time.time()
concepts = load_json("p13a_model.json")["audit"]["missing"]["concepts"]
kept = final_model()["names"]
check(sorted(kept) == sorted(COVERAGE_ORDER), f"the final model's terms {kept} differ from the concepts of the figure")
shares = {}
for c in COVERAGE_ORDER:
    s = concepts[c]["share_with_data"]
    shares[c] = (100 * s["uniform"], 100 * s["p_pop"], 100 * s["positives"])
y = np.arange(len(COVERAGE_ORDER))[::-1]
h = 0.26
fig, ax = plt.subplots(figsize=(WIDTH, 70 * MM), layout="constrained")
names = ["Random tracks (uniform negatives)", "Popularity-weighted tracks (p_pop negatives)",
         "Played tracks (training positives)"]
for k in range(3):
    bars = ax.barh(y + (1 - k) * h, [shares[c][k] for c in COVERAGE_ORDER], height=h, color=RAMP[k],
                   edgecolor=SURFACE, linewidth=0.6, label=names[k])
    label_bars(ax, bars, lambda v: f"{v:.0f}%")
ax.set_yticks(y, [CONCEPT_LABEL[c] for c in COVERAGE_ORDER])
ax.set_xlim(0, 112)
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_xlabel("Rows with data for the concept (percent)")
ax.grid(axis="y", visible=False)
fig.legend(loc="outside lower center", ncol=1)
save(fig, "fig_coverage", ["concept", "uniform_pct", "p_pop_pct", "positives_pct"],
     [[c, *shares[c]] for c in COVERAGE_ORDER])
finish("fig_coverage", t, Checks(),
       "Share of rows with data for each concept of the final model, for random tracks of the catalogue (uniform "
       "negatives), popularity-weighted tracks (p_pop negatives, Section 4.2) and played tracks (training "
       "positives). The more popular the tracks, the more often each kind of evidence exists.",
       "p13a_model.json (audit.missing.concepts)",
       "; ".join(f"{CONCEPT_LABEL[c]} {shares[c][0]:.1f} / {shares[c][1]:.1f} / {shares[c][2]:.1f}%"
                 for c in COVERAGE_ORDER))

## Figure 4.1: the loss decides what the popularity shape says

(a, b) The popularity term at the 5th to 95th percentiles of the positives' popularity, under Form A and Form B of the loss, for three samplers with 16 negatives each. Source: the arms of `p13b_model.json`, read at its stored shape quantiles.
(c) The largest gap of the explanation points to the reference fit, against the number of negatives m, for the mixture sampler. Source: the reference fits of `p13b2_model.json`. The Form A fit at m = 8,192 is the reference itself, so it is not drawn.

In [ ]:
# Figure 4.1: the popularity shape per loss form and sampler (p13b_model.json), and the gap to the reference
# against the number of negatives (p13b2_model.json).
SAMPLERS = (("uniform", "Uniform", "#2a78d6", "o"), ("p_pop", "Popularity-weighted", "#eb6834", "s"),
            ("mixture", "Mixture", "#1baf7a", "^"))
PCTS = (5, 25, 50, 75, 95)


def term_points(j, name, xq):
    """A term of a stored fit at the positions xq, as the stage reports read it."""
    T = np.asarray(j["tables"], dtype=np.float64)
    off = 0
    for t in j["terms"]:
        kn = np.asarray(t["knots"], dtype=np.float64)
        if t["name"] == name:
            tab = T[off:off + len(kn)]
            return [float(np.interp(np.clip(x, kn[0], kn[-1]), kn, tab)) for x in xq]
        off += len(kn) + 4
    raise FigureCheckError(f"the fit {j.get('arm')} has no term {name}")


t = time.time()
b_doc = load_json("p13b_model.json")
xq = b_doc["shape_quantiles"]["popularity"]
check(len(xq) == len(PCTS), "p13b_model.json holds no five shape quantiles of popularity")
arms_b = {j["arm"]: j for j in b_doc["arms"]}
ref = load_json("p13b2_model.json")["ref_fits"]
fig = plt.figure(figsize=(WIDTH, 60 * MM), layout="constrained")
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.05])
ax_a = fig.add_subplot(gs[0])
ax_b = fig.add_subplot(gs[1], sharey=ax_a)
ax_c = fig.add_subplot(gs[2])
out, spans = [], {}
for k, (ax, form) in enumerate([(ax_a, "A"), (ax_b, "B")]):
    for key, name, col, mark in SAMPLERS:
        check(f"{key}-{form}" in arms_b, f"p13b_model.json holds no arm {key}-{form}")
        vals = term_points(arms_b[f"{key}-{form}"], "popularity", xq)
        ax.plot(PCTS, vals, color=col, marker=mark, markersize=4, linewidth=1.5, label=name)
        out.append(["shape", f"{key}-{form}", *vals])
        spans[f"{key}-{form}"] = vals[-1] - vals[0]
    ax.axhline(0, color=INK_2, linewidth=0.6)
    ax.set_xticks(PCTS)
    ax.set_xlabel("Percentile of popularity")
    panel(ax, k, f"Form {form}")
ax_a.set_ylabel("Popularity term (log-odds)")
plt.setp(ax_b.get_yticklabels(), visible=False)
gaps32 = {}
for form, style in (("A", "--"), ("B", "-")):
    pts = sorted((int(r["m"]), float(r["gap_to_reference"])) for r in ref
                 if r["arm"] == f"mix32-{form}" and float(r["gap_to_reference"]) > 0)
    check(len(pts) >= 2, f"p13b2_model.json holds fewer than two reference fits of mix32-{form}")
    ms, gaps = zip(*pts)
    ax_c.plot(ms, gaps, linestyle=style, color=INK, marker="o", markersize=3.5, linewidth=1.2)
    ax_c.annotate(f"Form {form}", (ms[-1], gaps[-1]), xytext=(6, 0) if form == "A" else (-2, 7),
                  textcoords="offset points", fontsize=7, color=INK, va="center" if form == "A" else "bottom",
                  ha="left" if form == "A" else "right")
    out += [["gap", f"mix32-{form}", m, g, None, None, None] for m, g in pts]
    gaps32[form] = dict(pts).get(32)
ax_c.set_xscale("log", base=2)
ax_c.set_yscale("log")
m_all = sorted({int(r["m"]) for r in ref if str(r["arm"]).startswith("mix32-")})
ax_c.set_xticks(m_all[::2] if len(m_all) > 5 else m_all, [f"{m:,}" for m in (m_all[::2] if len(m_all) > 5 else m_all)])
ax_c.xaxis.set_minor_locator(ticker.NullLocator())
ax_c.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _p: f"{v:g}"))
ax_c.set_xlabel("Negatives per listener (m)")
ax_c.set_ylabel("Largest gap to the reference")
panel(ax_c, 2, "Gap to the reference")
fig.legend(*ax_a.get_legend_handles_labels(), loc="outside lower center", ncol=3, handlelength=1.8)
save(fig, "fig_loss_forms", ["kind", "arm", "p5_or_m", "p25_or_gap", "p50", "p75", "p95"], out)
finish("fig_loss_forms", t, Checks(),
       "How the loss form changes the popularity shape. (a, b) The popularity term at percentiles of the "
       "positives' popularity under Form A and Form B, for three samplers with 16 negatives each. (c) The largest "
       "gap between the explanation values of a fit and those of the reference fit with "
       f"{max(m_all):,} negatives, for the mixture sampler, against the number of negatives (both axes "
       "logarithmic).",
       "p13b_model.json (arms, shape_quantiles), p13b2_model.json (ref_fits)",
       "popularity p95 minus p5: " + ", ".join(f"{a} {v:+.1f}" for a, v in spans.items())
       + "; largest gap at m = 32: " + ", ".join(f"Form {f} {g:.2f}" for f, g in gaps32.items() if g is not None))

## Figure 4.2: the shapes of the final model

One panel per term, in the model's term order: the stored centred knot heights (linear between the knots, flat beyond the end knots, as the model scores), with the pointwise 95 percent band over the 100 listener bootstrap replicates, each centred on the background of the model's level.
A bounded term also shows the free fit of the same arm and penalty as a thin grey line. The points at the left are the no-match and no-data states with their bands.
The marks under each curve show the median, the 25th to 75th and the 5th to 95th percentiles of the term's value among the training positives on the shape.
The x axis is linear, or symmetric-logarithmic for a term with a long tail. Each panel has its own y scale, and its importance (the mean absolute contribution) is the number to compare across panels.

Checks: the rebuilt layout gives the stored raw tables, the centring gives the stored tables, and at the five shape quantiles and at the states the panel's band equals the stored C1 band of P13c, and the curve its stored value.

In [ ]:
# Figure 4.2: the shapes with bootstrap bands, from the final model's files (shared sources above).
GRID_POINTS = 400
TAIL_RATIO = 20.0
LINEAR_SHARE = 0.1
STATE_LABELS = {"zero": "no match", "missing": "no data"}
TERM_X = {"tag_affinity": "TF-IDF cosine of the track tags",
          "artist_tag_affinity": "TF-IDF cosine of the artist tags",
          "genre_affinity": "TF-IDF cosine of the genres",
          "artist_affinity": "Listening share of the track's artist",
          "colisten_same_artist": "Co-listening score, same artist",
          "audio_fit": "Audio fit (minus mean squared z-score)"}


def x_scale(knots):
    """("linear", None), or ("symlog", linear part) when the knots have a long tail."""
    a = np.abs(np.asarray(knots, dtype=np.float64))
    med, top = float(np.median(a)), float(a.max())
    if top > TAIL_RATIO * med:
        return "symlog", (LINEAR_SHARE * med if med > 0 else float(a[a > 0].min()))
    return "linear", None


def grid_of(knots, q, scale):
    """Every knot, the quantiles clamped into the knots, and GRID_POINTS evenly spaced points on the x scale."""
    kn = np.asarray(knots, dtype=np.float64)
    if scale[0] == "symlog":
        tr = mscale.SymmetricalLogTransform(10, scale[1], 1.0)
        a, b = tr.transform(np.array([kn[0], kn[-1]]))
        even = tr.inverted().transform(np.linspace(a, b, GRID_POINTS))
    else:
        even = np.linspace(kn[0], kn[-1], GRID_POINTS)
    qc = [min(max(float(x), kn[0]), kn[-1]) for x in q]
    return np.unique(np.concatenate([kn, np.asarray(qc, dtype=np.float64), np.clip(even, kn[0], kn[-1])])), qc


def term_values(t, off, T, x):
    """The term's value at x, as the model scores: linear between the knots, flat beyond the end knots."""
    kn = t.knot_array
    return np.interp(np.clip(x, kn[0], kn[-1]), kn, np.asarray(T)[off:off + t.K])


def shapes_data(chk):
    F = final_model()
    lay, stored, src = F["lay"], F["stored"], F["files"]["results"]
    bands, imp = F["doc"]["bootstrap"]["bands"], F["doc"]["bootstrap"]["importance"]
    terms = []
    for t, off in zip(lay.terms, lay.offsets):
        rows = {r["point"]: r for r in bands[t.name]}
        q = [float(x) for x in (F["shape_q"].get(t.name) or [])[:5]]
        scale = x_scale(t.knot_array)
        grid, qc = grid_of(t.knot_array, q, scale)
        curve = term_values(t, off, stored, grid)
        V = {s: np.array([term_values(t, off, T, grid) for T in F["boot"][s]]) for s in SCHEMES}
        band = {s: np.quantile(V[s], BAND, axis=0) for s in SCHEMES}
        for p, x in enumerate(qc):
            i = int(np.searchsorted(grid, x))
            r = rows.get(POINTS[p])
            check(r is not None and grid[i] == x, f"{t.name}: no stored C1 band at {POINTS[p]}")
            for s in SCHEMES:
                chk.close([band[s][0, i], band[s][1, i]], [r[s]["lo"], r[s]["hi"]],
                          f"{t.name}: the {s} band at {POINTS[p]}", src)
            chk.close(curve[i], r["value"], f"{t.name}: the curve at {POINTS[p]}", src, CENTER_TOL)
        free = None
        if F["free"] is not None and t.name in F["bounded"]:
            free = term_values(t, off, F["free"]["tables"], grid)
        states = []
        for key, flag, slot in (("zero", t.zero_state, gam.ZERO), ("missing", t.missing_state, gam.MISS)):
            if not flag:
                continue
            j = off + t.K + slot
            r = rows.get(key)
            check(r is not None, f"{t.name}: no stored C1 band for the {key} state")
            sb = {s: np.quantile(F["boot"][s][:, j], BAND) for s in SCHEMES}
            for s in SCHEMES:
                chk.close([sb[s][0], sb[s][1]], [r[s]["lo"], r[s]["hi"]], f"{t.name}: the {s} band of {key}", src)
            chk.close(stored[j], r["value"], f"{t.name}: the {key} state", src, CENTER_TOL)
            states.append({"key": key, "label": STATE_LABELS[key], "value": float(stored[j]),
                           "lo": float(sb["listener"][0]), "hi": float(sb["listener"][1]),
                           "lo2": float(sb["twoway"][0]), "hi2": float(sb["twoway"][1]),
                           "free": None if free is None else float(F["free"]["tables"][j])})
        terms.append({"name": t.name, "label": CONCEPT_LABEL.get(t.name, t.name),
                      "xlabel": TERM_X.get(t.name, t.source),
                      "scale": scale, "grid": grid, "curve": curve, "lo": band["listener"][0],
                      "hi": band["listener"][1], "lo2": band["twoway"][0], "hi2": band["twoway"][1], "free": free,
                      "states": states, "q": qc, "importance": float(imp[t.name])})
    return {"terms": terms, "n_listener": int(len(F["boot"]["listener"])), "n_twoway": int(len(F["boot"]["twoway"])),
            "free_name": None if F["free"] is None else F["free"]["name"]}


def symlog_ticks(lo, hi, lt):
    """0 and the powers of ten outside the linear part."""
    out = [0.0] if lo <= 0.0 <= hi else []
    top = max(abs(lo), abs(hi))
    for k in range(math.ceil(math.log10(lt)), math.floor(math.log10(top)) + 1):
        out += [x for x in (-(10.0 ** k), 10.0 ** k) if lo <= x <= hi]
    return sorted(out)


def draw_shapes(S):
    """A fixed layout in mm, so the panels of a column line up: per term a state axis, then the curve axis."""
    terms = S["terms"]
    ncol = 2
    nrow = math.ceil(len(terms) / ncol)
    W, row_h, legend_h = 150.0, 47.0, 13.0
    H = 1.0 + row_h * nrow + legend_h
    col_x, sw, gp, cw, ah = (14.0, 88.0), 9.5, 3.0, 47.5, 26.0

    def rect(x, y_top, w, h):
        return (x / W, 1.0 - (y_top + h) / H, w / W, h / H)

    fig = plt.figure(figsize=(W * MM, H * MM))
    blue = color("gam")
    for k, T in enumerate(terms):
        r, c = divmod(k, ncol)
        y0 = 1.0 + r * row_h + 6.0
        ax_s = fig.add_axes(rect(col_x[c], y0, sw, ah))
        ax = fig.add_axes(rect(col_x[c] + sw + gp, y0, cw, ah), sharey=ax_s)
        ax.fill_between(T["grid"], T["lo"], T["hi"], color=blue, alpha=0.22, lw=0, zorder=1)
        if T["free"] is not None:
            ax.plot(T["grid"], T["free"], color=MUTED, lw=0.8, zorder=2)
        ax.plot(T["grid"], T["curve"], color=blue, lw=1.3, zorder=3)
        ax.axhline(0.0, color=RULE, lw=0.6, zorder=0)
        if T["scale"][0] == "symlog":
            ax.set_xscale("symlog", linthresh=T["scale"][1], linscale=1.0)
            ax.xaxis.set_major_locator(ticker.FixedLocator(symlog_ticks(T["grid"][0], T["grid"][-1], T["scale"][1])))
            ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _p: "0" if v == 0 else f"{v:g}"))
            ax.xaxis.set_minor_locator(ticker.NullLocator())
        ax.set_xlim(T["grid"][0], T["grid"][-1])
        ax.grid(axis="x", visible=False)
        ax.tick_params(axis="y", labelleft=False, length=0)
        ax.spines["left"].set_visible(False)
        ax.tick_params(axis="x", labelsize=7)
        ax.set_xlabel(T["xlabel"], fontsize=7.5)
        ax_s.set_title(f"({LETTERS[k]}) {T['label']}", loc="left", fontweight="bold", fontsize=8)
        ax.set_title(f"importance {T['importance']:.2f}", loc="right", color=INK_2, fontsize=7, fontweight="normal")
        q = T["q"]
        if len(q) == 5:
            tr = ax.get_xaxis_transform()
            ax.plot([q[0], q[4]], [0.07, 0.07], transform=tr, color=INK_2, lw=0.8, solid_capstyle="butt", zorder=4)
            ax.plot([q[1], q[3]], [0.07, 0.07], transform=tr, color=INK_2, lw=2.6, solid_capstyle="butt", zorder=4)
            ax.plot([q[2]], [0.07], transform=tr, marker="o", ms=3.4, color=SURFACE, mec=INK_2, mew=0.9, ls="none",
                    zorder=5)
        for i, s in enumerate(T["states"]):
            ax_s.vlines([i], [s["lo"]], [s["hi"]], color=blue, lw=1.0, zorder=3)
            ax_s.plot([i], [s["value"]], marker="o", ms=3.2, color=blue, mec=SURFACE, mew=0.5, ls="none", zorder=4)
            if s["free"] is not None:
                ax_s.plot([i + 0.34], [s["free"]], marker="o", ms=2.4, color=MUTED, ls="none", zorder=2)
        ns = len(T["states"])
        ax_s.set_xlim(-0.6, max(ns, 1) - 0.25)
        ax_s.set_xticks(list(range(ns)))
        ax_s.set_xticklabels([s["label"] for s in T["states"]], rotation=90, fontsize=6.5)
        ax_s.axhline(0.0, color=RULE, lw=0.6, zorder=0)
        ax_s.grid(axis="x", visible=False)
        ax_s.tick_params(axis="y", labelsize=7)
        if k % ncol == 0:
            ax_s.set_ylabel("Centred effect (log-odds)", fontsize=7.5)
        vals = [T["lo"], T["hi"], T["curve"]] + ([T["free"]] if T["free"] is not None else [])
        lo = min(float(np.min(v)) for v in vals)
        hi = max(float(np.max(v)) for v in vals)
        for s in T["states"]:
            lo = min(lo, s["lo"], s["free"] if s["free"] is not None else lo)
            hi = max(hi, s["hi"], s["free"] if s["free"] is not None else hi)
        rng = (hi - lo) or 1.0
        ax_s.set_ylim(lo - 0.22 * rng, hi + 0.08 * rng)
    handles = [Line2D([], [], color=blue, lw=1.3), Patch(facecolor=blue, alpha=0.22, lw=0),
               Line2D([], [], color=INK_2, lw=2.2, marker="o", ms=3.2, mfc=SURFACE, mec=INK_2)]
    names = ["Final model", f"95% band, {S['n_listener']} listener bootstrap replicates",
             "Training positives: median, 25-75% (thick), 5-95% (thin)"]
    if S["free_name"] is not None:
        handles.insert(2, Line2D([], [], color=MUTED, lw=0.8))
        names.insert(2, "Free fit without bounds")
    fig.legend(handles, names, loc="lower center", bbox_to_anchor=(0.5, 1.5 / H), ncol=2, handlelength=2.2,
               columnspacing=1.8)
    return fig


t = time.time()
chk = Checks()
S = shapes_data(chk)
rows = []
for T in S["terms"]:
    free = T["free"] if T["free"] is not None else [None] * len(T["grid"])
    rows += [[T["name"], "curve", float(x), float(T["curve"][i]), float(T["lo"][i]), float(T["hi"][i]),
              float(T["lo2"][i]), float(T["hi2"][i]), None if free[i] is None else float(free[i])]
             for i, x in enumerate(T["grid"])]
    rows += [[T["name"], s["key"], None, s["value"], s["lo"], s["hi"], s["lo2"], s["hi2"], s["free"]]
             for s in T["states"]]
save(draw_shapes(S), "fig_shapes",
     ["term", "kind", "x", "value", "listener_lo", "listener_hi", "twoway_lo", "twoway_hi", "free_fit"], rows)
values = []
for T in S["terms"]:
    at = {p: c1_row(T["name"], p) for p in ("p5", "p50", "p95", "zero", "missing")}
    values.append(f"{T['label']}: p5/p50/p95 " + " / ".join(f"{at[p]['value']:+.3f}" for p in ("p5", "p50", "p95"))
                  + "".join(f", {STATE_LABELS[p]} {at[p]['value']:+.3f}" for p in ("zero", "missing") if at[p])
                  + f", importance {T['importance']:.3f}")
finish("fig_shapes", t, chk,
       f"Shape functions of the final model, in log-odds points, centred on the popularity-weighted negatives of "
       f"its training data. The blue line is the fitted shape with its pointwise 95% band over {S['n_listener']} "
       f"listener bootstrap refits"
       + (", and the grey line is the same model fitted without monotone bounds" if S["free_name"] else "")
       + ". The points at the left are the no-match and no-data states. The marks under each curve show the "
         "median, the middle half and the 5th to 95th percentiles of the concept's values among the training "
         "positives with a match. Each panel has its own vertical scale, and its importance is the mean absolute "
         "contribution of the term.",
       f"{final_model()['files']['results']}, {final_model()['files']['arrays']}, p13a_model.json, "
       f"p13d_model_set.npz, scratch/p13c_<tag>/fits.json (background means)",
       "; ".join(values))

## Figure 4.3: suppression between the two artist terms

Artist affinity in the final model and in the model without co-listening (the cold-track GAM), and same-artist co-listening in the final model, at the 5th to 95th percentiles of each concept's values among the training positives.
Source: `p14_cold_gam.json` (`shapes`). The final model's two lines are checked against its stored C1 values.

In [ ]:
# Figure 4.3: the two artist terms, from p14_cold_gam.json, checked against the final model's C1 values.
t = time.time()
chk = Checks()
shp = load_json("p14_cold_gam.json")["shapes"]
idx = [shp["points"].index(p) for p in POINTS[:5]]
lines = [("Same-artist co-listening, final model", "final", "colisten_same_artist", color("gam"), "--"),
         ("Artist affinity, final model", "final", "artist_affinity", color("gam"), "-"),
         ("Artist affinity, model without co-listening", "cold", "artist_affinity", color("cold_gam"), "-")]
fig, ax = plt.subplots(figsize=(WIDTH, 62 * MM), layout="constrained")
out = []
for name, model, term, col, style in lines:
    vals = [float(shp[model][term][i]) for i in idx]
    if model == "final":
        chk.close(vals, [c1_row(term, p)["value"] for p in POINTS[:5]], f"{term} of the final model",
                  final_model()["files"]["results"], 1e-9)
    ax.plot(PCTS, vals, color=col, linestyle=style, marker="o", markersize=4, linewidth=1.6, label=name)
    ax.annotate(f"{vals[-1]:+.1f}", (PCTS[-1], vals[-1]), xytext=(5, 0), textcoords="offset points", va="center",
                fontsize=7, color=INK)
    out.append([name, *vals])
ax.axhline(0, color=INK_2, linewidth=0.6)
ax.set_xticks(PCTS)
ax.set_xlim(2, 102)
ax.set_xlabel("Percentile of the concept's value among training positives")
ax.set_ylabel("Contribution (log-odds)")
ax.legend(loc="upper left")
save(fig, "fig_artist_suppression", ["line", "p5", "p25", "p50", "p75", "p95"], out)
finish("fig_artist_suppression", t, chk,
       "Suppression between the two artist terms. Artist affinity in the final model (solid blue) and in the "
       "model without co-listening (pink), and same-artist co-listening in the final model (dashed blue), at "
       "percentiles of each concept's values among the training positives.",
       "p14_cold_gam.json (shapes)",
       "; ".join(f"{r[0]}: p5 {r[1]:+.3f}, p50 {r[3]:+.3f}, p95 {r[5]:+.3f}" for r in out))

## Figure 6.1: accuracy on the held-out listeners

(a) NDCG@100 of every arm (Table 6.1). (b, c) The GAM minus each arm with 95 percent intervals: the confirmatory tests C1 to C4 (Table 6.2) and the exploratory tests on NDCG@100. The shaded band is the SESOI.
Source: `p14.json` (`arms`, `tests`). Checks: every metric of Table 6.1 and every plotted difference, computed again from the ranks in `p14.npz`.

In [ ]:
# Figure 6.1: accuracy and the tests, from p14.json, checked against the ranks in p14.npz.
t = time.time()
chk = Checks()
doc14, z14 = p14_doc(), p14_arrays()
lm = {}
for a in P14_ARMS:
    r = np.asarray(z14[f"rank_{a}"])
    check(r.ndim == 2 and r.shape[1] == 2, f"p14.npz: rank_{a} is not (listeners, 2)")
    lm[a] = listener_metrics(r)
    st = doc14["arms"][a]
    chk.same(len(r), int(st["listeners"]), f"listeners of {a}", "p14.json")
    chk.close([lm[a][k].mean() for k in METRICS], [st[k] for k in METRICS], f"metrics of {a} (Table 6.1)",
              "p14.json")
    chk.close([np.median(r[:, p].astype(np.float64)) for p in range(2)],
              [st["types"][ty]["median_rank"] for ty in PICK_TYPES], f"median ranks of {a}", "p14.json")
tests = [(x["id"], x["arm"], x["difference"]) for x in doc14["tests"]["confirmatory"]]
tests += [(x["id"], x["arm"], x["difference"]) for x in doc14["tests"]["exploratory"] if x["metric"] == "ndcg_at_100"]
for tid, a, d in tests:
    chk.close(list(paired(lm["gam"]["ndcg_at_100"], lm[a]["ndcg_at_100"]).values()), triple(d),
              f"{tid}, the GAM minus {a} (Table 6.2)", "p14.json")
sesoi = float(doc14["tests"]["sesoi"])
acc = {a: float(doc14["arms"][a]["ndcg_at_100"]) for a in P14_ARMS}
fig = plt.figure(figsize=(WIDTH, 80 * MM), layout="constrained")
gs = fig.add_gridspec(2, 2, width_ratios=[1.05, 1], height_ratios=[3, 4])
ax_a = fig.add_subplot(gs[:, 0])
order = sorted(acc, key=acc.get)
bars = ax_a.barh(range(len(order)), [acc[a] for a in order], color=[color(a) for a in order], edgecolor=SURFACE,
                 linewidth=0.6, height=0.72)
label_bars(ax_a, bars, lambda v: f"{v:.4f}")
ax_a.set_yticks(range(len(order)), [label(a) for a in order])
top = max(acc.values())
ax_a.set_xlim(0, top * 1.35)
ax_a.xaxis.set_major_locator(ticker.MaxNLocator(4))
ax_a.set_xlabel("NDCG@100")
ax_a.grid(axis="y", visible=False)
panel(ax_a, 0, "All arms")
near = [x for x in tests if abs(num(x[2]["mean"])) < 0.01]
far = [x for x in tests if abs(num(x[2]["mean"])) >= 0.01]
for k, (spec, group, name) in enumerate([(gs[0, 1], near, "Close comparisons"), (gs[1, 1], far, "Large comparisons")]):
    ax = fig.add_subplot(spec)
    group = sorted(group, key=lambda x: num(x[2]["mean"]))
    ax.axvspan(-sesoi, sesoi, color="#eeeeee", zorder=0, lw=0)
    ax.axvline(0, color=INK_2, linewidth=0.6)
    for yy, (tid, a, d) in enumerate(group):
        m, lo, hi = triple(d)
        ax.errorbar(m, yy, xerr=[[m - lo], [hi - m]], fmt="o", color=color(a), ecolor=color(a), elinewidth=1.4,
                    capsize=2.5, markersize=4.5)
    ax.set_yticks(range(len(group)), [f"{label(a)} ({tid})" for tid, a, _ in group])
    ax.set_ylim(-0.6, len(group) - 0.4)
    if group:
        lo_all = min(num(d["lo"]) for _, _, d in group)
        hi_all = max(num(d["hi"]) for _, _, d in group)
        if k == 0:  # close comparisons: symmetric around 0, the SESOI band inside
            lim = 1.15 * max(abs(lo_all), abs(hi_all), sesoi)
            ax.set_xlim(-lim, lim)
        else:
            left, right = min(0.0, lo_all), max(0.0, hi_all)
            ax.set_xlim(left - 0.08 * (right - left), right + 0.08 * (right - left))
    ax.xaxis.set_major_locator(ticker.MaxNLocator(2 if k == 0 else 3, symmetric=k == 0))
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _p: "0" if abs(v) < 1e-12 else f"{v:+.3f}"))
    ax.set_xlabel("GAM minus arm (NDCG@100)")
    ax.grid(axis="y", visible=False)
    panel(ax, k + 1, name)
save(fig, "fig_accuracy", ["kind", "arm", "value", "lo", "hi"],
     [["ndcg100", a, acc[a], None, None] for a in order]
     + [[f"difference {tid}", a, *triple(d)] for tid, a, d in tests])
finish("fig_accuracy", t, chk,
       "Accuracy on the evaluation listeners. (a) NDCG@100 of every arm. (b, c) The GAM minus each arm, with 95% "
       f"intervals over listeners. The grey band marks the SESOI of {sesoi:.5f} on either side of zero. C1 to C4 "
       "are the confirmatory tests, E1 to E3 the exploratory tests on NDCG@100.",
       "p14.json (arms, tests), checked against p14.npz",
       "; ".join(f"{label(a)} {acc[a]:.5f}" for a in sorted(acc, key=acc.get, reverse=True)))

## Figure 6.2: where the accuracy comes from

(a) NDCG@100 by popularity band of the pick (Table 6.3). (b) NDCG@100 for picks with and without a same-artist co-listening neighbour in the listener's history.
Source: `p14.json` (`slices`). Checks: every plotted value and pick count, computed again from the ranks, popularity centiles and neighbour flags in `p14.npz`.

In [ ]:
# Figure 6.2: two slices of P14, from p14.json, checked against p14.npz.
SLICE_ARMS = ("gam", "artist_only", "knn_same", "als_d", "popularity")
POP_BANDS = ("01-10", "11-50", "51-90", "91-100")


def pop_bands(centile):
    c = np.asarray(centile)
    return np.where(c <= 10, 0, np.where(c <= 50, 1, np.where(c <= 90, 2, 3)))


t = time.time()
chk = Checks()
doc14, z14 = p14_doc(), p14_arrays()
pm = {a: pick_metrics(z14[f"rank_{a}"])["ndcg_at_100"] for a in SLICE_ARMS}
codes = {"popularity band": pop_bands(z14["pop_centile"]),
         "same-artist neighbour": np.where(np.asarray(z14["neighbour"], dtype=bool), 0, 1)}
groups = {}
for sl, cd in codes.items():
    rows_sl = doc14["slices"][sl]
    out = []
    for c, row in enumerate(rows_sl):
        mask = cd == c
        chk.same(int(mask.sum()), int(row["picks"]), f"picks of {sl} {row['level']}", "p14.json")
        vals = {}
        for a in SLICE_ARMS:
            got = cluster_mean(pm[a], mask)
            chk.close([got["mean"], got["lo"], got["hi"]], triple(row["arms"][a]["ndcg_at_100"]),
                      f"{sl} {row['level']}, {a}", "p14.json")
            vals[a] = float(row["arms"][a]["ndcg_at_100"]["mean"])
        out.append((row["level"], int(row["picks"]), vals))
    groups[sl] = out
check([g[0] for g in groups["popularity band"]] == list(POP_BANDS), "p14.json holds other popularity bands")
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(WIDTH, 68 * MM), layout="constrained",
                                 gridspec_kw={"width_ratios": [1.6, 1]})
w = 0.16
out = []
top = max(v for g in groups.values() for _, _, vals in g for v in vals.values())
for k, (ax, sl, labels_x, name) in enumerate([
        (ax_a, "popularity band", [f"{g[0]}\n{g[1]:,} picks" for g in groups["popularity band"]], "By popularity band"),
        (ax_b, "same-artist neighbour", [("With" if g[0].startswith("with ") else "Without") + f"\n{g[1]:,} picks"
                                         for g in groups["same-artist neighbour"]], "Same-artist evidence")]):
    x = np.arange(len(groups[sl]))
    for j, a in enumerate(SLICE_ARMS):
        vals = [g[2][a] for g in groups[sl]]
        ax.bar(x + (j - 2) * w, vals, width=w, color=color(a), edgecolor=SURFACE, linewidth=0.5, label=label(a))
        out += [[sl, g[0], a, g[2][a]] for g in groups[sl]]
    ax.set_xticks(x, labels_x)
    ax.set_ylim(0, top * 1.25)
    ax.grid(axis="x", visible=False)
    panel(ax, k, name)
ax_a.set_ylabel("NDCG@100")
without = groups["same-artist neighbour"][1][2]
for j, a in enumerate(SLICE_ARMS):
    ax_b.annotate(f"{without[a]:.4f}", (1 + (j - 2) * w, without[a]), xytext=(0, 3), textcoords="offset points",
                  rotation=90, ha="center", va="bottom", fontsize=6.3, color=INK)
fig.legend(*ax_a.get_legend_handles_labels(), loc="outside lower center", ncol=5, handlelength=1.2,
           columnspacing=1.2)
save(fig, "fig_slices", ["slice", "level", "arm", "ndcg100"], out)
nb = groups["same-artist neighbour"]
finish("fig_slices", t, chk,
       "NDCG@100 by slice. (a) By popularity band of the pick, with the number of picks per band. (b) For picks "
       "with and without a same-artist co-listening neighbour in the listener's history.",
       "p14.json (slices), checked against p14.npz",
       f"without a neighbour: {nb[1][1]:,} of {nb[0][1] + nb[1][1]:,} picks; "
       + ", ".join(f"{label(a)} {nb[1][2][a]:.5f}" for a in SLICE_ARMS)
       + f"; GAM with a neighbour {nb[0][2]['gam']:.5f}")

## Figure 6.3: beyond accuracy

(a) The depth of the first new artist: for each arm, the share of listeners whose best-ranked track by a new artist stands at or above a rank, on a log axis (the full curve behind the median of Table 6.4).
(b) The cap trade-off: the change in NDCG@10 against no cap, with 95 percent intervals, against the mean number of distinct artists in the top 10, for caps of 5, 3, 2 and 1 tracks per artist (Table 6.5).
Sources: `beyond.npz` (`first_new_<arm>`) and `beyond.json` (`results.arms`). Checks: the percentiles and shares of the first new artist, and every plotted cap value, computed again from `beyond.npz`, and the uncapped NDCG@10 against Table 6.1 (`p14.json`).

In [ ]:
# Figure 6.3: the first new artist (beyond.npz) and the cap trade-off (beyond.json), checked against beyond.npz.
BEYOND_ARMS = ("gam", "artist_only", "cold_gam", "knn_same", "knn_both", "als_d", "popularity")
CAP_ORDER = ("none", "5", "3", "2", "1")  # from the loosest to the tightest

t = time.time()
chk = Checks()
res_b = load_json("beyond.json")["results"]["arms"]
check(sorted(res_b) == sorted(BEYOND_ARMS), f"beyond.json holds the arms {sorted(res_b)}")
zb = load_npz("beyond.npz", ["caps"] + [f"first_new_{a}" for a in BEYOND_ARMS]
              + [f"{a}_cap_{k}" for a in BEYOND_ARMS for k in ("ndcg10", "distinct")])
col_of = {("none" if int(c) == 0 else str(int(c))): j for j, c in enumerate(zb["caps"])}
check(sorted(col_of) == sorted(CAP_ORDER), f"beyond.npz holds the caps {zb['caps'].tolist()}")
first, caps, cdf_rows, medians = {}, {}, [], {}
for a in BEYOND_ARMS:
    r, src = res_b[a], "beyond.json"
    fn = np.asarray(zb[f"first_new_{a}"], dtype=np.int64)
    has = fn >= 1
    q = rank_quantiles(fn[has])
    for p, v in zip(QUANTILES, q):
        chk.same(v, r["first_new"]["quantiles"][f"p{p}"], f"{a}: first new artist, p{p} (Table 6.4)", src)
    chk.same(int(has.sum()), int(r["first_new"]["listeners_with_new"]), f"{a}: listeners with a new artist", src)
    for c in ("10", "100"):
        got = interval((has & (fn <= int(c))).astype(np.float64))
        chk.close([got["mean"], got["lo"], got["hi"]], triple(r["first_new"]["within"][c]),
                  f"{a}: first new artist within {c}", src)
    first[a] = (np.sort(fn[has]), len(fn))
    medians[a] = q[2]
    nd = np.asarray(zb[f"{a}_cap_ndcg10"], dtype=np.float64)
    di = np.asarray(zb[f"{a}_cap_distinct"], dtype=np.float64)
    base = nd[:, col_of["none"]]
    chk.close(base.mean(), r["caps"]["none"]["ndcg_at_10"]["mean"], f"{a}: NDCG@10 without a cap (Table 6.5)", src)
    chk.close(base.mean(), p14_doc()["arms"][a]["ndcg_at_10"], f"{a}: NDCG@10 without a cap against Table 6.1",
              "p14.json")
    chk.close(di[:, col_of["none"]].mean(), r["distinct_artists_top10"]["mean"],
              f"{a}: distinct artists in the top 10 without a cap (Table 6.4)", src)
    caps[a] = []
    for c in CAP_ORDER:
        j, rc = col_of[c], r["caps"][c]
        d = paired(nd[:, j], base)
        chk.close([d["mean"], d["lo"], d["hi"]], triple(rc["ndcg_minus_none"]),
                  f"{a}: cap {c}, change in NDCG@10 (Table 6.5)", src)
        chk.close(di[:, j].mean(), rc["distinct_artists"]["mean"], f"{a}: cap {c}, distinct artists", src)
        caps[a].append((c, float(rc["distinct_artists"]["mean"]), *triple(rc["ndcg_minus_none"])))

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(WIDTH, 78 * MM), layout="constrained")
x_max = max(int(f[0][-1]) for f in first.values() if len(f[0]))
grid = np.unique(np.concatenate([np.arange(1, 101), np.geomspace(100, max(x_max, 101), 400).round()]))
for a in BEYOND_ARMS:
    xs, n = first[a]
    share = np.searchsorted(xs, grid, side="right") / n
    ax_a.step(grid, share, where="post", color=color(a), lw=1.3, label=label(a))
    cdf_rows += [["first_new", a, float(g), float(s)] for g, s in zip(grid, share)]
for c in (10, 100):
    ax_a.axvline(c, color=RULE, lw=0.6, zorder=0)
rank_ticks(ax_a, 1, max(x_max, 10))
ax_a.set_xlim(1, max(x_max, 10) * 1.2)
ax_a.set_ylim(0, 1.0)
ax_a.yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
ax_a.set_xlabel("Rank of the first track by a new artist")
ax_a.set_ylabel("Listeners, cumulative share")
panel(ax_a, 0, "First new artist")
for a in BEYOND_ARMS:
    pts = caps[a]
    xs_c = [p[1] for p in pts]
    ys_c = [p[2] for p in pts]
    err = [[p[2] - p[3] for p in pts], [p[4] - p[2] for p in pts]]
    ax_b.plot(xs_c, ys_c, color=color(a), lw=1.1, zorder=2)
    ax_b.errorbar(xs_c, ys_c, yerr=err, fmt="o", color=color(a), ecolor=color(a), elinewidth=0.9, capsize=1.6,
                  markersize=3.4, mec=SURFACE, mew=0.5, zorder=3)
    cdf_rows += [["cap", a, p[0], p[1], p[2], p[3], p[4]] for p in pts]
for c, xx, yy, *_ in caps["gam"]:
    ax_b.annotate("no cap" if c == "none" else f"cap {c}", (xx, yy), xytext=(0, -9) if c in ("none", "1") else (0, 7),
                  textcoords="offset points", ha="center", va="center", fontsize=6.3, color=INK, zorder=5,
                  path_effects=[patheffects.withStroke(linewidth=2.2, foreground=SURFACE)])
ax_b.axhline(0, color=INK_2, lw=0.6, zorder=1)
ax_b.xaxis.set_major_locator(ticker.MultipleLocator(2))
ax_b.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _p: "0" if abs(v) < 1e-12 else f"{v:+.3f}"))
ax_b.set_xlabel("Distinct artists in the top 10 (mean)")
ax_b.set_ylabel("Change in NDCG@10 against no cap")
panel(ax_b, 1, "Cap per artist")
fig.legend(*ax_a.get_legend_handles_labels(), loc="outside lower center", ncol=4)
save(fig, "fig_beyond", ["part", "arm", "rank_or_cap", "share_or_artists", "change", "lo", "hi"], cdf_rows)
g2 = dict((p[0], p) for p in caps["gam"])["2"]
finish("fig_beyond", t, chk,
       f"The top-10 lists of each arm beyond accuracy ({first['gam'][1]:,} listeners). (a) The cumulative share of "
       "listeners whose first track by a new artist stands at or above each rank (logarithmic axis), the curve "
       "behind the median of Table 6.4. (b) The change in NDCG@10 against no cap, with 95% intervals over "
       "listeners, against the mean number of distinct artists in the top 10. From left to right, each line passes "
       "no cap and caps of 5, 3, 2 and 1 tracks per artist (Table 6.5).",
       "beyond.npz (first_new_<arm>, <arm>_cap_ndcg10, <arm>_cap_distinct), beyond.json (results.arms)",
       "first new artist, median rank: " + ", ".join(f"{label(a)} {medians[a]:,}" for a in BEYOND_ARMS)
       + f"; GAM cap 2: {g2[2]:+.5f} ({g2[3]:+.5f} to {g2[4]:+.5f}), {g2[1]:.2f} artists")

## Figure 6.4: the cold-start curve

NDCG@100 and R@100 against the history length N for the four arms of the curve, with the stored 95 percent intervals as error bars. The x axis is logarithmic, and the full-history point stands apart at the right, with no line to the curve. Popularity ignores the history, so it is a flat line with its interval as a band. The y axis is logarithmic, because the values span more than a decade.
Source: `cold_start_curve.json` (`results`). Checks: every mean and interval (Table 6.6), computed again from the ranks in `cold_start_curve.npz`.

In [ ]:
# Figure 6.4: the cold-start curve, from cold_start_curve.json, checked against cold_start_curve.npz.
CURVE_ARMS = ("gam", "cold_gam", "knn_both")
COLD_ARMS = CURVE_ARMS + ("popularity",)
COLD_METRICS = ("ndcg_at_100", "recall_at_100")
COLD_STYLE = {"gam": ("o", -0.025), "knn_both": ("s", 0.0), "cold_gam": ("^", 0.025), "popularity": (None, 0.0)}

t = time.time()
chk = Checks()
cdoc = load_json("cold_start_curve.json")
res_c = cdoc["results"]
zc = load_npz("cold_start_curve.npz", ["points"] + [f"rank_{a}" for a in CURVE_ARMS]
              + [f"p14_rank_{a}" for a in COLD_ARMS])
pts = [int(x) for x in zc["points"]]
check(pts + ["full"] == res_c["points"], "the cut points of cold_start_curve.npz differ from its JSON file")
curve = {}
for a in COLD_ARMS:
    got = []
    for j in range(len(pts) + 1):
        if j == len(pts):
            r = zc[f"p14_rank_{a}"]
        else:
            r = zc["p14_rank_popularity"] if a == "popularity" else zc[f"rank_{a}"][:, j, :]
        m = listener_metrics(r)
        got.append({k: interval(m[k]) for k in COLD_METRICS})
    for g, row in zip(got, res_c["arms"][a]):
        for k in COLD_METRICS:
            chk.close([g[k]["mean"], g[k]["lo"], g[k]["hi"]], triple(row[k]),
                      f"{a} at N = {row['n']}, {k} (Table 6.6)", "cold_start_curve.json")
    curve[a] = [{k: triple(row[k]) for k in COLD_METRICS} for row in res_c["arms"][a]]
xs = [math.log10(n) for n in pts]
xfull = xs[-1] + 0.55
fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 66 * MM), layout="constrained")
for k, (ax, metric, name) in enumerate(zip(axes, COLD_METRICS, ("NDCG@100", "R@100"))):
    for a in COLD_ARMS:
        marker, dx = COLD_STYLE[a]
        rows_a = curve[a]
        if a == "popularity":
            m, lo, hi = rows_a[-1][metric]
            span = [xs[0] - 0.12, xfull + 0.12]
            ax.fill_between(span, [lo, lo], [hi, hi], color=color(a), alpha=0.15, lw=0, zorder=1)
            ax.plot(span, [m, m], color=color(a), lw=1.0, zorder=2)
            continue
        m = np.array([r[metric][0] for r in rows_a])
        lo = np.array([r[metric][1] for r in rows_a])
        hi = np.array([r[metric][2] for r in rows_a])
        x = np.array(xs + [xfull]) + dx
        ax.plot(x[:-1], m[:-1], color=color(a), lw=1.3, zorder=3)
        ax.errorbar(x, m, yerr=[m - lo, hi - m], fmt=marker, ms=3.6, color=color(a), ecolor=color(a), elinewidth=0.9,
                    capsize=1.8, mec=SURFACE, mew=0.6, zorder=4)
    ax.axvline((xs[-1] + xfull) / 2.0, color=RULE, lw=0.6, zorder=0)
    ax.set_yscale("log")
    pos = [v for a in COLD_ARMS for r in curve[a] for v in r[metric] if v > 0]
    ax.set_ylim(min(pos) / 1.4, max(pos) * 1.4)
    ax.yaxis.set_major_locator(ticker.LogLocator(subs=(1.0, 2.0, 5.0)))
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _p: f"{v:g}"))
    ax.yaxis.set_minor_locator(ticker.NullLocator())
    ax.set_xticks(xs + [xfull], [str(n) for n in pts] + ["full"])
    ax.set_xlim(xs[0] - 0.12, xfull + 0.12)
    ax.grid(axis="x", visible=False)
    ax.set_xlabel("Training pairs kept per listener (N)")
    panel(ax, k, name)
handles = [Line2D([], [], color=color(a), lw=1.3 if COLD_STYLE[a][0] else 1.0, marker=COLD_STYLE[a][0], ms=3.6,
                  mec=SURFACE, mew=0.6) for a in COLD_ARMS]
fig.legend(handles, [label(a) + (" (no history)" if a == "popularity" else "") for a in COLD_ARMS],
           loc="outside lower center", ncol=4, handlelength=2.2, columnspacing=1.6)
save(fig, "fig_cold_start", ["arm", "metric", "N", "mean", "lo", "hi"],
     [[a, k, n, *r[k]] for a in COLD_ARMS for n, r in zip(pts + ["full"], curve[a]) for k in COLD_METRICS])
g = [r["ndcg_at_100"][0] for r in curve["gam"]]
finish("fig_cold_start", t, chk,
       f"(a) NDCG@100 and (b) R@100 of {int(cdoc['data']['listeners']):,} evaluation listeners when each history "
       "keeps only its first N training pairs, with 95% intervals over listeners. The points at the right use the "
       "full history. Popularity ignores the history. Both axes are logarithmic.",
       "cold_start_curve.json (results), checked against cold_start_curve.npz",
       "GAM NDCG@100 at N = " + ", ".join(f"{n} {v:.5f}" for n, v in zip(pts + ["full"], g))
       + f"; N = {pts[-1]} reaches {100 * g[-2] / g[-1]:.1f}% of the full history")

## Figure 6.5: which reasons reach the listener

For each reason on the GAM's top 10: the share of tracks where it is a candidate (its concept matched), where it passes the bootstrap rule D1, and where it is shown after the model-choice rule S4, with its mean contribution and odds factor when shown (Table 6.7).
Source: `p14.json` (`reasons.rows`). Checks: every plotted share and mean, computed again from the reason arrays of `p14.npz`.

In [ ]:
# Figure 6.5: the reasons on the GAM's top 10, from p14.json, checked against p14.npz.
REASON_ORDER = ("artist", "artist_tag_affinity", "tag_affinity", "audio_fit", "genre_affinity")

t = time.time()
chk = Checks()
doc14, z14 = p14_doc(), p14_arrays()
rs = doc14["reasons"]
names_r = [r["reason"] for r in rs["rows"]]
check(sorted(names_r) == sorted(REASON_ORDER), f"p14.json holds the reasons {names_r}")
cand, d1, shown, val = (np.asarray(z14[k]) for k in ("candidate", "shown_d1", "shown", "reason_value"))
check(cand.shape[0] == len(names_r), "the reason arrays of p14.npz do not hold one row per reason")
for g, r in enumerate(rs["rows"]):
    sg = shown[g].astype(bool)
    have = [cand[g].mean(), d1[g].mean(), sg.mean(), val[g][sg].mean() if sg.any() else math.nan,
            np.exp(val[g][sg].mean()) if sg.any() else math.nan]
    chk.close(have, [r["candidate"], r["shown_d1"], r["shown"], num(r["mean_shown"]), num(r["odds_of_mean"])],
              f"the reason {r['reason']} (Table 6.7)", "p14.json", 1e-9)
chk.same(int(np.prod(shown.shape[1:])), int(rs["tracks"]), "tracks of the top 10", "p14.json")
rows_r = sorted(rs["rows"], key=lambda r: REASON_ORDER.index(r["reason"]))
y = np.arange(len(rows_r))[::-1]
h = 0.26
fig, ax = plt.subplots(figsize=(WIDTH, 64 * MM), layout="constrained")
stages = (("candidate", "Candidate (its concept matched)"), ("shown_d1", "Passes the bootstrap rule D1"),
          ("shown", "Shown (passes D1 and S4)"))
for k, (key, name) in enumerate(stages):
    bars = ax.barh(y + (1 - k) * h, [100 * r[key] for r in rows_r], height=h, color=RAMP[k], edgecolor=SURFACE,
                   linewidth=0.6, label=name)
    if key == "shown":
        label_bars(ax, bars, lambda v: f"{v:.1f}%")


def odds_text(x):
    return f"{x:,.0f}" if x >= 100 else f"{x:.2f}"


for yy, r in zip(y, rows_r):
    if r["mean_shown"] is None:
        continue
    ax.annotate(f"mean {r['mean_shown']:+.3f} (odds ×{odds_text(r['odds_of_mean'])})", (1.0, yy),
                xycoords=("axes fraction", "data"), xytext=(6, 0), textcoords="offset points", va="center",
                ha="left", fontsize=6.8, color=INK_2)
ax.set_yticks(y, [REASON_LABEL[r["reason"]] for r in rows_r])
ax.set_xlim(0, 112)
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_xlabel(f"Tracks of the top 10 (percent of {int(rs['tracks']):,} tracks)")
ax.grid(axis="y", visible=False)
fig.legend(loc="outside lower center", ncol=3)
save(fig, "fig_reasons", ["reason", "candidate_pct", "d1_pct", "shown_pct", "mean_value_shown", "odds_factor_of_mean"],
     [[r["reason"], 100 * r["candidate"], 100 * r["shown_d1"], 100 * r["shown"], r["mean_shown"], r["odds_of_mean"]]
      for r in rows_r])
finish("fig_reasons", t, chk,
       f"Reasons on the GAM's top 10 for the evaluation listeners ({int(rs['tracks']):,} tracks). For each reason, "
       "the share of tracks where its concept matched, where it passes the bootstrap rule D1, and where it is shown "
       "after the model-choice rule S4, with its mean contribution and odds factor when shown.",
       "p14.json (reasons.rows), checked against p14.npz",
       "; ".join(f"{REASON_LABEL[r['reason']]} shown {100 * r['shown']:.1f}%" for r in rows_r))

## Figure 6.6: removing the evidence of the shown reasons

For the tracks of the GAM's top 10, each counterfactual removes the evidence that a shown reason cites (or keeps only the artist evidence) and ranks the full catalogue again with the frozen model.
(a) The share of tested tracks that leave the top 10, with 95 percent intervals for listener clusters. Row 3 measures sufficiency, so its bar is one minus the share that stay. (b) The new rank: median, 25th to 75th and 5th to 95th percentiles, on a log axis. The rows follow Table 6.8.
Sources: `evidence_removal.json` (`results`) and `evidence_removal.npz` (the new ranks). Checks: every count, share, interval and percentile, computed again from `evidence_removal.npz`.
When the stage has not run, the cell skips this figure.

In [ ]:
# Figure 6.6: the evidence-removal counterfactuals, from evidence_removal.json and evidence_removal.npz.
ER_ROWS = (  # (row of Table 6.8, label, place in results, measure)
    ("1", "Listed co-listened tracks removed", ("artist_shown",), "leave"),
    ("2", "All history tracks by the artist removed", ("artist_all",), "leave"),
    ("3", "Only the history tracks by the artist kept", ("artist_kept",), "stay"),
    ("4", "Cited tags and genres removed", ("content_all",), "leave"),
    ("", "Artist tags", ("content", "artist_tag_affinity"), "leave"),
    ("", "Track tags", ("content", "tag_affinity"), "leave"),
    ("", "Genres", ("content", "genre_affinity"), "leave"),
    ("5", "No audio data", ("audio",), "leave"),
    ("6", "Evidence of every shown reason removed", ("list",), "leave"),
)
ER_KEYS = ("rank_shown", "rank_all", "rank_kept", "rank_content", "rank_audio", "rank_list")
TOP_K = 10


def er_ranks(z, place, content):
    """The (listeners, tracks) rank codes of one row: >= 1 a new rank, 0 not tested, -1 no history left."""
    kind = place[0]
    if kind == "content":
        return z["rank_content"][:, content.index(place[1]), :]
    if kind == "content_all":
        return z["rank_content"].reshape(len(z["rank_content"]), -1)
    return z[{"artist_shown": "rank_shown", "artist_all": "rank_all", "artist_kept": "rank_kept",
              "audio": "rank_audio", "list": "rank_list"}[kind]]


t = time.time()
if not ((MODEL_DIR / "evidence_removal.json").is_file() and (MODEL_DIR / "evidence_removal.npz").is_file()):
    skip("fig_evidence_removal", "evidence_removal.json or evidence_removal.npz is missing. Run "
                                 "python3 -m pipeline.runner --only evidence_removal, then this notebook again")
else:
    chk = Checks()
    edoc = load_json("evidence_removal.json")
    content = list(edoc["data"]["content"])
    ze = load_npz("evidence_removal.npz", list(ER_KEYS))
    er = []
    for row, name, place, measure in ER_ROWS:
        res = edoc["results"]
        for key in place:
            res = res[key]
        what = f"row {row or '4'} {name}"
        chk.same(res["measure"], measure, f"{what}: measure", "evidence_removal.json")
        r = np.asarray(er_ranks(ze, place, content), dtype=np.int64)
        tested = r >= 1
        hit = (r > TOP_K) if measure == "leave" else (r <= TOP_K)
        got = cluster_mean(hit, tested)
        got_ci = [got["mean"], max(0.0, got["lo"]), min(1.0, got["hi"])]
        chk.same(int(tested.sum()), int(res["tested"]), f"{what}: tracks tested (Table 6.8)", "evidence_removal.json")
        chk.close(got_ci, triple(res["share"]), f"{what}: share (Table 6.8)", "evidence_removal.json")
        q = rank_quantiles(r[tested])
        for p, v in zip(QUANTILES, q):
            chk.same(v, res["rank"][f"p{p}"], f"{what}: new rank p{p} (Table 6.8)", "evidence_removal.json")
        m, lo, hi = triple(res["share"])
        leave = (m, lo, hi) if measure == "leave" else (1 - m, 1 - hi, 1 - lo)
        er.append({"row": row, "name": name, "measure": measure, "tested": int(res["tested"]), "share": (m, lo, hi),
                   "leave": leave, "q": q})
    n_rows = len(er)
    y = np.arange(n_rows)[::-1].astype(float)
    fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(WIDTH, 84 * MM), layout="constrained", sharey=True,
                                     gridspec_kw={"width_ratios": [1, 1]})
    blue = color("gam")
    for yy, e in zip(y, er):
        m, lo, hi = e["leave"]
        ax_a.barh(yy, 100 * m, height=0.62, color=blue if e["row"] else RAMP[0], edgecolor=SURFACE, linewidth=0.6)
        ax_a.errorbar(100 * m, yy, xerr=[[100 * (m - lo)], [100 * (hi - m)]], fmt="none", ecolor=INK, elinewidth=0.8,
                      capsize=1.6)
        txt = f"{100 * m:.1f}%" + (f" ({100 * e['share'][0]:.1f}% stay)" if e["measure"] == "stay" else "")
        ax_a.annotate(txt, (100 * hi, yy), xytext=(3, 0), textcoords="offset points", va="center", ha="left",
                      fontsize=6.5, color=INK)
        q = e["q"]
        if q[0] is None:
            continue
        ink = blue if e["row"] else RAMP[0]
        ax_b.plot([q[0], q[4]], [yy, yy], color=ink, lw=0.9, solid_capstyle="butt")
        ax_b.plot([q[1], q[3]], [yy, yy], color=ink, lw=3.2, solid_capstyle="butt")
        ax_b.plot([q[2]], [yy], marker="o", ms=3.6, color=SURFACE, mec=INK, mew=0.9, ls="none", zorder=4)
        ax_b.annotate(f"{q[2]:,}", (1.0, yy), xycoords=("axes fraction", "data"), xytext=(4, 0),
                      textcoords="offset points", va="center", ha="left", fontsize=6.5, color=INK)
    ax_a.set_yticks(y, [(f"{e['row']}  " if e["row"] else "      ") + e["name"] for e in er])
    ax_a.set_xlim(0, 135)
    ax_a.set_xticks([0, 25, 50, 75, 100])
    ax_a.set_xlabel("Tested tracks (percent)")
    ax_a.grid(axis="y", visible=False)
    panel(ax_a, 0, "Leave the top 10")
    q_max = max(e["q"][4] for e in er if e["q"][0] is not None)
    ax_b.axvline(TOP_K + 0.5, color=INK_2, lw=0.6, zorder=0)
    ax_b.annotate("rank 10", (TOP_K + 0.5, 0.0), xycoords=("data", "axes fraction"), xytext=(3, 3),
                  textcoords="offset points", ha="left", va="bottom", fontsize=6.3, color=INK_2)
    ax_b.annotate("median", (1.0, 1.0), xycoords="axes fraction", xytext=(4, 2), textcoords="offset points",
                  ha="left", va="bottom", fontsize=6.5, color=INK_2)
    rank_ticks(ax_b, 1, max(q_max * 3, 1e3), every=2)
    ax_b.set_xlim(0.8, max(q_max * 3, 1e3))
    ax_b.set_xlabel("New rank of the tested tracks")
    ax_b.grid(axis="y", visible=False)
    ax_b.tick_params(axis="y", length=0)
    panel(ax_b, 1, "New rank")
    for i in range(1, n_rows):  # a rule between the counterfactuals of Table 6.8
        if er[i]["row"]:
            for ax in (ax_a, ax_b):
                ax.axhline(y[i] + 0.5, color=GRID, lw=0.6, zorder=0)
    ax_a.set_ylim(-0.6, n_rows - 0.4)
    save(fig, "fig_evidence_removal",
         ["row", "counterfactual", "measure", "tested", "share", "share_lo", "share_hi", "leave", "leave_lo",
          "leave_hi", "rank_p5", "rank_p25", "rank_p50", "rank_p75", "rank_p95"],
         [[e["row"], e["name"], e["measure"], e["tested"], *e["share"], *e["leave"], *e["q"]] for e in er])
    finish("fig_evidence_removal", t, chk,
           f"Removing the evidence of the shown reasons, for the GAM's top 10 of "
           f"{int(edoc['data']['listeners']):,} listeners. (a) The share of tested tracks that leave the top 10, "
           "with 95% intervals over listeners. Row 3 keeps only the artist evidence, and its label adds the share "
           "that stay. (b) The new rank of the tested tracks: median (point), 25th to 75th percentile (thick line) "
           "and 5th to 95th percentile (thin line), on a logarithmic axis. The vertical line marks rank 10. The rows "
           "follow Table 6.8.",
           "evidence_removal.json (results), evidence_removal.npz (rank_*)",
           "; ".join(f"{e['row'] or '4'} {e['name']}: {100 * e['leave'][0]:.1f}% leave, median {e['q'][2]:,}"
                     for e in er if e["q"][0] is not None))

## Figure 6.7: the tree-based audit of the shapes

Each panel shows one term at the training positives' percentiles and at its no-match and no-data states, for the boosted tree model (its value with the range over its bags), the free spline fit and the bounded final model (each with its 95 percent listener band).
Source: `tree_audit.json` (`points.rows`). Checks: the final model's points and bands against its stored C1 values.

In [ ]:
# Figure 6.7: the tree-GAM audit, from tree_audit.json, checked against the final model's C1 values.
TERM_ORDER = ("colisten_same_artist", "artist_affinity", "artist_tag_affinity", "tag_affinity", "genre_affinity",
              "audio_fit")
POINT_LABEL = ("5", "25", "50", "75", "95", "NM", "ND")

t = time.time()
chk = Checks()
tdoc = load_json("tree_audit.json")
n_bags = len(tdoc.get("bags") or []) or int(tdoc["settings"]["bags"])
FITS = ((f"Tree model, range over {n_bags} bags", "tree", color("knn_same"), -0.22),
        ("Free spline fit, 95% band", "free", color("als_p6"), 0.0),
        ("Final model, 95% band", "final", color("gam"), 0.22))
src = final_model()["files"]["results"]
fig, axes = plt.subplots(2, 3, figsize=(WIDTH, 104 * MM), layout="constrained")
out = []
for k, (ax, term) in enumerate(zip(axes.flat, TERM_ORDER)):
    pts_t = [r for r in tdoc["points"]["rows"] if r["term"] == term and r["point"] in POINTS]
    xs = np.array([POINTS.index(r["point"]) for r in pts_t], dtype=float)
    for name, key, col, dx in FITS:
        if key == "tree":
            vals = [(r["tree"], r["tree_bags"][0], r["tree_bags"][1]) for r in pts_t]
        else:
            vals = [(r[key]["value"], r[key]["lo"], r[key]["hi"]) for r in pts_t]
        if key == "final":
            for r, v in zip(pts_t, vals):
                c1 = c1_row(term, r["point"])
                check(c1 is not None, f"{term}: no stored C1 value at {r['point']}")
                chk.close(list(v), [c1["value"], c1["listener"]["lo"], c1["listener"]["hi"]],
                          f"{term} at {r['point']}, the final model's column", src)
        m = np.array([v[0] for v in vals])
        lo = np.array([v[1] for v in vals])
        hi = np.array([v[2] for v in vals])
        # the band as a line from lo to hi: a percentile band need not hold the fit, so no error bar of signed lengths
        ax.vlines(xs + dx, lo, hi, color=col, lw=0.9, zorder=2)
        ax.plot(xs + dx, m, "o", markersize=2.8, color=col, zorder=3, label=name)
        out += [[term, r["point"], key, *v] for r, v in zip(pts_t, vals)]
    ax.axhline(0, color=INK_2, linewidth=0.5)
    ax.axvline(4.5, color=GRID, linewidth=0.8)
    ax.set_xticks(range(len(POINTS)), POINT_LABEL, fontsize=6.3)
    ax.tick_params(axis="y", labelsize=6.8)
    ax.grid(axis="x", visible=False)
    panel(ax, k, CONCEPT_LABEL[term])
for ax in axes[:, 0]:
    ax.set_ylabel("Log-odds")
for ax in axes[1, :]:
    ax.set_xlabel("Percentile or state", fontsize=7.5)
handles, names = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, names, loc="outside lower center", ncol=3)
save(fig, "fig_tree_audit", ["term", "point", "fit", "value", "lo", "hi"], out)
falls = tdoc.get("falls") or {}
finish("fig_tree_audit", t, chk,
       "The tree-based audit. Each panel shows one term at percentiles of the training positives' values and at "
       "its no-match (NM) and no-data (ND) states, for the boosted tree model (with the range over its "
       f"{words(n_bags)} bags), the free spline fit and the bounded final model (each with its 95% listener band).",
       "tree_audit.json (points.rows)",
       "tree model, change from p5 to p95: "
       + ", ".join(f"{CONCEPT_LABEL[k2]} {v['net']:+.2f}" for k2, v in falls.items() if k2 in CONCEPT_LABEL))

## Figure 6.8: what the catalogue keeps of a recent history

(a) The ten most played artist names of the ListenBrainz listens (share of all listens), marked by whether they are among the 15 largest artists of the matched history. (b) The 15 largest artists of the matched history (share of the history weight).
Source: `demo/<user>/lb_demo.json` (`match_artists`, `history.top_artists`), the output of the ListenBrainz demo. It holds the listener's own tracks and stays on this computer.

In [ ]:
# Figure 6.8: the raw and the matched artists of the demo history, from demo/<user>/lb_demo.json.
@functools.lru_cache(maxsize=None)
def demo_doc():
    root = MODEL_DIR / "demo"
    if DEMO_USER:
        path = root / DEMO_USER / "lb_demo.json"
        check(path.is_file(), f"{path} is missing")
    else:
        found = sorted(root.glob("*/lb_demo.json")) if root.is_dir() else []
        check(len(found) == 1, f"{len(found)} demo folders under {root}: set DEMO_USER to one of them")
        path = found[0]
    with open(path, encoding="utf-8") as f:
        return json.load(f)


t = time.time()
dd = demo_doc()
raw = dd["match_artists"]["raw"]
top15 = dd["history"]["top_artists"]
n_listens = int(dd["match_artists"]["listens"])
kept15 = {str(a.get("name") or "").casefold() for a in top15}
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(WIDTH, 74 * MM), layout="constrained")
names_a = [str(a["name"]) for a in raw][::-1]
vals_a = [100 * float(a["share"]) for a in raw][::-1]
in15 = [n.casefold() in kept15 for n in names_a]
bars = ax_a.barh(names_a, vals_a, color=[color("gam") if k else color("knn_same") for k in in15], edgecolor=SURFACE,
                 linewidth=0.6, height=0.72)
label_bars(ax_a, bars, lambda v: f"{v:.1f}%")
ax_a.set_xlim(0, max(vals_a) * 1.3)
ax_a.set_xlabel("Share of all listens (percent)")
ax_a.grid(axis="y", visible=False)
panel(ax_a, 0, "Raw listens")
names_b = [str(a.get("name") or a["artist"]) for a in top15][::-1]
vals_b = [100 * float(a["share"]) for a in top15][::-1]
bars = ax_b.barh(names_b, vals_b, color=color("gam"), edgecolor=SURFACE, linewidth=0.6, height=0.72)
label_bars(ax_b, bars, lambda v: f"{v:.1f}%")
ax_b.set_xlim(0, max(vals_b) * 1.3)
ax_b.set_xlabel("Share of the matched history (percent)")
ax_b.grid(axis="y", visible=False)
panel(ax_b, 1, "Matched history")
fig.legend(handles=[Patch(color=color("gam"), label=f"Among the {len(top15)} largest matched artists"),
                    Patch(color=color("knn_same"), label="Not among them")], loc="outside lower center", ncol=2)
save(fig, "fig_demo_match", ["panel", "artist", "share_pct", f"in_matched_top{len(top15)}"],
     [["raw", n, v, k] for n, v, k in zip(names_a[::-1], vals_a[::-1], in15[::-1])]
     + [["matched", n, v, True] for n, v in zip(names_b[::-1], vals_b[::-1])])
n_out = sum(1 for k in in15 if not k)
finish("fig_demo_match", t, Checks(),
       "What a catalogue that ends in 2013 keeps of a recent history. "
       f"(a) The {words(len(raw))} most played artists of "
       f"the demonstration listener's {n_listens:,} ListenBrainz listens, marked by whether they are among the "
       f"{len(top15)} largest artists of the matched history. (b) The {len(top15)} largest artists of the matched "
       "history.",
       "demo/<user>/lb_demo.json (match_artists, history.top_artists)",
       f"{words(n_out)} of the {words(len(raw))} most played artists are not among the {len(top15)} largest "
       "matched artists")

## Figure 6.9: example explanations

Three tracks of the demo's GAM list: the track at rank 1, then the next tracks by rank whose artist differs from the artist of every earlier example. For each track, one bar per shown reason (the demo keeps only the reasons that D1 and S4 show) with its contribution in log-odds and its odds factor exp(value). The panels share one x axis, so the size of the artist reason against the content reasons stays visible. The candidate reasons that the rules do not show are named under the bars.
Source: `demo/<user>/lb_demo.json` (`gam`, `model`). Checks: the demo names the final model with its numbers of replicates and S3 members, and each shown reason is positive, in descending order, with odds factor exp(value).

In [ ]:
# Figure 6.9: the example explanations, from demo/<user>/lb_demo.json, checked against the final model's files.
EXAMPLES = 3
TITLE_CHARS = 70


def artist_key(r):
    for x in r["reasons"]:
        if x["reason"] == "artist" and x.get("credit") is not None:
            return ("credit", int(x["credit"]))
    return ("name", r.get("artist"))


def odds_label(x):
    if x is None or not math.isfinite(x):
        return "very large"
    return f"×{x:,.0f}" if x >= 100 else (f"×{x:,.2f}" if x < 10 else f"×{x:,.1f}")


t = time.time()
chk = Checks()
dd, F = demo_doc(), final_model()
want = {"final": F["rec"]["fit_name"], "replicates": int(len(F["boot"]["listener"])), "members": len(F["members"])}
for k, v in want.items():
    chk.same(dd["model"].get(k), v, f"the demo's model, {k}", "the final model's files")
ex, seen = [], set()
for r in sorted(dd["gam"], key=lambda r: int(r["rank"])):
    key = artist_key(r)
    if ex and key in seen:
        continue
    ex.append(r)
    seen.add(key)
    if len(ex) == EXAMPLES:
        break
check(ex, "lb_demo.json holds no GAM list")
for r in ex:
    vals = [x["value"] for x in r["reasons"]]
    check(all(v > 0 for v in vals) and vals == sorted(vals, reverse=True),
          f"rank {r['rank']}: the shown reasons are not positive in descending order")
    for x in r["reasons"]:
        try:
            e = math.exp(x["value"])
        except OverflowError:
            e = math.inf
        chk.same(x["odds"], e, f"rank {r['rank']}, {x['reason']}: odds factor exp(value)", "lb_demo.json")
        ev = x.get("evidence") or {}
        check(int(ev.get("count") or 0) >= len(ev.get("units") or []),
              f"rank {r['rank']}: an evidence count is below its units")
sizes = [max(1, len(e["reasons"]))
         + (0.6 if [c for c in e.get("candidates", []) if c not in [x["reason"] for x in e["reasons"]]] else 0.0)
         for e in ex]
H = sum(9.0 + 6.0 * s for s in sizes) + 12.0
fig = plt.figure(figsize=(WIDTH, H * MM), layout="constrained")
gs = fig.add_gridspec(len(ex), 1, height_ratios=[s + 0.9 for s in sizes])
top = max(x["value"] for e in ex for x in e["reasons"])
axes, out = [], []
for i, e in enumerate(ex):
    ax = fig.add_subplot(gs[i, 0], sharex=axes[0] if axes else None)
    axes.append(ax)
    shown_r = [x["reason"] for x in e["reasons"]]
    not_shown = [c for c in e.get("candidates", []) if c not in shown_r]
    vals = [x["value"] for x in e["reasons"]]
    yy = np.arange(len(vals))[::-1].astype(float)
    ax.barh(yy, vals, height=0.55, color=color("gam"), zorder=2)
    for y0, x in zip(yy, e["reasons"]):
        ax.text(x["value"] + 0.012 * top, y0, f"{x['value']:+.2f} (odds {odds_label(x['odds'])})", va="center",
                ha="left", fontsize=6.8, color=INK)
        out.append([int(e["rank"]), int(e["item"]), e.get("title"), e.get("artist"), x["reason"], float(x["value"]),
                    float(x["odds"])])
    ax.set_yticks(yy, [REASON_LABEL.get(x["reason"], x["reason"]) for x in e["reasons"]])
    ax.set_ylim(-1.3 if not_shown else -0.7, len(vals) - 0.3)
    track = e.get("title") or f"item {e['item']}"
    title = f"({LETTERS[i]}) {e['rank']}. “{track}” by {e.get('artist') or 'unknown artist'}"
    ax.set_title(title if len(title) <= TITLE_CHARS else title[:TITLE_CHARS - 1] + "…", loc="left", fontweight="bold",
                 fontsize=8)
    if not_shown:
        ax.text(top * 1.34, -0.95, "candidate reasons that the rules do not show: "
                + ", ".join(REASON_LABEL.get(c, c) for c in not_shown), ha="right", va="center", fontsize=6.5,
                color=MUTED)
    ax.grid(axis="y", visible=False)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0)
    if i < len(ex) - 1:
        ax.tick_params(axis="x", labelbottom=False)
axes[0].set_xlim(0.0, top * 1.34)
axes[-1].set_xlabel("Contribution to the track's log-odds against the mean candidate")
save(fig, "fig_explanations", ["rank", "item", "title", "artist", "reason", "value", "odds"], out)
finish("fig_explanations", t, chk,
       f"The reasons shown for {words(len(ex))} of the GAM's "
       + ("top 25 tracks" if max(int(e["rank"]) for e in ex) <= 25 else "listed tracks")
       + " for the demonstration history (ranks "
       + ", ".join(str(e["rank"]) for e in ex[:-1]) + f" and {ex[-1]['rank']}). Each bar is the contribution of a "
       "reason to the log-odds of the track against the mean candidate, with its odds factor. Only the reasons that "
       "pass both stability rules appear.",
       "demo/<user>/lb_demo.json (gam, model)",
       "; ".join(f"rank {e['rank']}: " + ", ".join(f"{REASON_LABEL.get(x['reason'], x['reason'])} {x['value']:+.3f} "
                                                   f"({odds_label(x['odds'])})" for x in e["reasons"]) for e in ex))

## Captions and summary

`captions.md` lists every figure in thesis order with its number, file, caption, sources and the values that the text quotes.

In [ ]:
# captions.md in figures/notebook, and the summary of the run.
lines = ["# Figure captions", "",
         "The figures of the thesis in thesis order, as drawn by the figures notebook. Each figure is in this folder "
         "as PNG (300 dpi), PDF and SVG, with its plotted values in a CSV file of the same name. The thesis links "
         "the PNG files.", ""]
for n, name in FIGURES:
    r = RESULTS.get(name)
    lines += [f"## Figure {n}: {name}.png", ""]
    if r is None:
        lines += ["Not drawn: its cell did not run.", ""]
    elif r["status"] == "skipped":
        lines += [f"Not drawn: {r['reason']}.", ""]
    else:
        lines += [f"Figure {n}: {r['caption']}", "", f"Sources: {r['sources']}.", ""]
        if r["values"]:
            lines += [f"Values for the text: {r['values']}.", ""]
text = "\n".join(lines)
with open(OUT / "captions.md", "w", newline="", encoding="utf-8") as f:
    f.write(text)
drawn = [name for _, name in FIGURES if RESULTS.get(name, {}).get("status") == "drawn"]
skipped = [name for _, name in FIGURES if RESULTS.get(name, {}).get("status") == "skipped"]
print(f"{len(drawn)} of {len(FIGURES)} figures drawn in {time.time() - T0:.1f} s"
      + (f", skipped: {', '.join(skipped)}" if skipped else "") + f". Captions: {OUT / 'captions.md'}")